# Backward Kolmogorov Transport

Retained manuscript experiments: OU, double/multiwell systems, independent
products, and alanine dipeptide. Dependencies: Python 3.11, NumPy 2.4.6,
SciPy 1.17.1, Matplotlib 3.11.1, threadpoolctl 3.6.0, Numba 0.64.0.

Run from the project root. Supplementary runners and `experiment_store.py`
are required; data are unpacked for the notebook session and repacked on exit.
The main training sizes are frozen. The explicit rank, time, particle-count,
sample-size and coefficient-regime studies are retained manuscript diagnostics.
All prescribed seeds, including unstable results, are included.

The main 2D double-well experiments and equal-data comparison share the
admissible source with harmonic-coordinate variance 0.3; both use supplementary runners. The equal-data comparison retains BKT and KDE particle flow only.
The nine-well limitation remains. The 10D/50D products exploit
separability and do not demonstrate general coupled high-dimensional sampling.

For figures only, run **Figure data and style** and the desired plotting cell.
All results are in [the single report](outputs/all_experiment_results_and_assessment.md).
[Reproduction commands](outputs/all_experiment_results_and_assessment.md#files-and-reproduction) include the standalone alanine
runner. The manuscript source is not rewritten by notebook execution.


In [ ]:
# Fixed manuscript settings and verified data archives
import sys, os, time, json, math
# Keep numerical outputs and cache together, outside the source files.
from pathlib import Path
_here = Path.cwd().resolve()
PROJECT_DIR = _here.parent if _here.name == 'outputs' and (_here.parent / 'BKT_experiments.ipynb').exists() else _here
import sys
sys.path.insert(0, str(PROJECT_DIR / "supplementary"))
from experiment_store import activate_notebook_store
activate_notebook_store()
OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
os.chdir(OUTPUT_DIR)
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
import numpy as np
import matplotlib; import matplotlib.pyplot as plt
QUICK = False   # Full experiment budgets
if QUICK: NPAIR, M2D, M10D, SEEDS_MAIN, SEEDS_SEC, SEEDS_10D, DT = 3000, 120, 60, 1, 1, 1, 0.1   # small sizes for a quick check of the code (a few minutes)
else:     NPAIR, M2D, M10D, SEEDS_MAIN, SEEDS_SEC, SEEDS_10D, DT = 200000, 2000, 1000, 10, 10, 10, 0.05
DT_A2 = 0.05  # transport step for all 2D A double-well experiments
TUNE = {'wfrac': 0.075, 'rbf_n1d': 30, 'poly_p1d': 20, 'poly_p2d': 16}   # fixed dictionary settings
# Existing 10D defaults remain n=8 (657 functions), p=3 (286 functions).
R10 = 16  # retained nonconstant modes in Part I 10D experiments
# Retained appendix sensitivity experiments.
TRAINING_BUDGETS = [10000, 100000, 200000]
FIXED_TRAINING_PAIRS = {'A2_beta0.0':200000, 'A2_beta0.25':200000, 'A2_beta0.5':200000, 'A2_beta1.0':200000, 'B_quartic4':200000, 'B_poly9':200000, 'A10_beta0.0':100000}
APPENDIX_SYSTEMS = {}
RETAINED_SYSTEMS = {f'A2_beta{b}' for b in [0.,.25,.5,1.]} | {'B_quartic4','B_poly9','A10_beta0.0'}
REFERENCE_REPEATS = 10
CACHE_ENABLED = True

# Embedded persistent cache: no local Python modules are required.
"""Content-addressed numerical cache. Arrays only; never loads pickled code.

Callers must include numerical parameters and implementation/dependency hashes.
Cache hits return fresh arrays, so subsequent in-place edits cannot poison disk.
"""
from pathlib import Path
from collections import Counter
from functools import wraps
import hashlib
import json
import os
import sys
import tempfile
import types
import zipfile
import numpy as np
import scipy

CACHE_DIR = Path('cache/numerics')
STATS = Counter()


def _identity(value):
    if isinstance(value, np.ndarray):
        a = np.ascontiguousarray(value)
        return {'array': hashlib.sha256(a.tobytes()).hexdigest(),
                'shape': a.shape, 'dtype': a.dtype.str}
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): _identity(v) for k, v in sorted(value.items(), key=lambda p: str(p[0]))}
    if isinstance(value, (list, tuple)):
        return [_identity(v) for v in value]
    if isinstance(value, types.CodeType):
        return {'bytecode': value.co_code.hex(), 'consts': _identity(value.co_consts),
                'names': value.co_names, 'vars': value.co_varnames,
                'freevars': value.co_freevars, 'cellvars': value.co_cellvars}
    if isinstance(value, bytes):
        return {'bytes': value.hex()}
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    raise TypeError(f'Unsupported cache key: {type(value).__name__}')


def digest(value):
    return hashlib.sha256(json.dumps(_identity(value), sort_keys=True,
                                     separators=(',', ':')).encode()).hexdigest()


def code_hash(*functions):
    """Ignores notebook execution counts/filenames; includes defaults and code."""
    parts = []
    for fn in functions:
        fn = getattr(fn, '__wrapped__', fn)
        parts.append((fn.__code__, fn.__defaults__, fn.__kwdefaults__))
    return digest(parts)


def class_hash(cls):
    functions = []
    for _, value in sorted(vars(cls).items()):
        if isinstance(value, (staticmethod, classmethod)):
            value = value.__func__
        if isinstance(value, types.FunctionType):
            functions.append(value)
    return code_hash(*functions)


def _pack(value, arrays):
    if isinstance(value, np.ndarray):
        if value.dtype.hasobject:
            raise TypeError('Object arrays are not supported')
        key = f'a{len(arrays)}'; arrays[key] = value
        return {'array': key}
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, dict):
        return {'dict': [[k, _pack(v, arrays)] for k, v in value.items()]}
    if isinstance(value, (tuple, list)):
        return {'tuple' if isinstance(value, tuple) else 'list': [_pack(v, arrays) for v in value]}
    if value is None or isinstance(value, (str, int, float, bool)):
        return {'scalar': value}
    raise TypeError(f'Unsupported cache value: {type(value).__name__}')


def _unpack(tree, arrays):
    if 'array' in tree: return arrays[tree['array']].copy()
    if 'dict' in tree: return {k: _unpack(v, arrays) for k, v in tree['dict']}
    if 'tuple' in tree: return tuple(_unpack(v, arrays) for v in tree['tuple'])
    if 'list' in tree: return [_unpack(v, arrays) for v in tree['list']]
    return tree['scalar']


def cached_arrays(name, key, compute):
    if not CACHE_ENABLED:
        STATS[f'{name}:disabled'] += 1
        return compute()
    config = {'format': 1, 'python': sys.version_info[:2], 'numpy': np.__version__,
              'scipy': scipy.__version__, 'parameters': key}
    folder = CACHE_DIR / name
    path = folder / (digest(config) + '.npz')
    if path.exists():
        try:
            with np.load(path, allow_pickle=False) as data:
                result = _unpack(json.loads(str(data['metadata'])), data)
            STATS[f'{name}:hit'] += 1
            return result
        except (ValueError, OSError, KeyError, EOFError, zipfile.BadZipFile):
            STATS[f'{name}:invalid'] += 1
    result = compute()
    arrays = {}; tree = _pack(result, arrays)
    arrays['metadata'] = np.array(json.dumps(tree))
    arrays['config'] = np.array(json.dumps(_identity(config), sort_keys=True))
    folder.mkdir(parents=True, exist_ok=True)
    # Unique temporary file and atomic replacement also allow concurrent readers.
    with tempfile.NamedTemporaryFile(dir=folder, suffix='.npz', delete=False) as f:
        temporary = Path(f.name)
        np.savez_compressed(f, **arrays)
    os.replace(temporary, path)
    STATS[f'{name}:miss'] += 1
    return result


def cache_grid_1d(fn):
    @wraps(fn)
    def wrapped(Vfun, N=1401, L=3.5):
        grid = np.linspace(-L, L, N)
        values = np.asarray(Vfun(grid))
        return cached_arrays('grid_1d', dict(N=N, L=L, potential=values, code=code_hash(fn)),
                             lambda: fn(Vfun, N=N, L=L))
    return wrapped


os.makedirs('figures', exist_ok=True); os.makedirs('tables', exist_ok=True)

def section(cell, title, notes=''):
    """Print a short experiment heading without creating a duplicate log"""
    print('\n' + '=' * 100 + f'\n[code cell {cell}] {title}\n' + (notes + '\n' if notes else '') + '-' * 100)
section(1, 'Settings', f'QUICK={QUICK}; pairs per data set {NPAIR}; particles M={M2D} (2D), {M10D} (10D); seeds {SEEDS_MAIN}/{SEEDS_SEC}/{SEEDS_10D} (main/secondary/10D cells); transport step {DT} (2D A: {DT_A2}); dictionaries {TUNE}')

# Enforce the declared thread setting even in an already initialized kernel.
os.environ['BLIS_NUM_THREADS'] = '1'
from threadpoolctl import threadpool_limits
_NOTEBOOK_THREAD_LIMITS = threadpool_limits(limits=1)


In [ ]:
# code cell 2: one-dimensional helpers (grid eigenpairs of the double well, bump source, transport in one dimension)
"""One-dimensional helpers: grid eigenpairs of the double well, bump source, support for separable multidimensional products."""
import numpy as np
from scipy.linalg import eigh
from scipy.stats import norm

# ---------------- Hermite (OU) basis ----------------
def hermite_basis(x, r):
    x = np.asarray(x, float)
    P = np.zeros((r + 1, x.size)); dP = np.zeros((r + 1, x.size))
    P[0] = 1.0
    if r >= 1: P[1] = x
    for k in range(1, r):
        P[k + 1] = (x * P[k] - np.sqrt(k) * P[k - 1]) / np.sqrt(k + 1)
    for k in range(1, r + 1):
        dP[k] = np.sqrt(k) * P[k - 1]
    return P, dP


# ---------------- 1D generator on a grid (Dirichlet form) ----------------
@cache_grid_1d
def setup_1d(Vfun, N=1401, L=3.5):
    xg = np.linspace(-L, L, N); h = xg[1] - xg[0]
    pi = np.exp(-Vfun(xg)); pi /= np.sum(pi) * h
    pih = np.sqrt(pi[:-1] * pi[1:])
    A = np.zeros((N, N)); i = np.arange(N - 1)
    A[i, i] += pih / h; A[i + 1, i + 1] += pih / h; A[i, i + 1] -= pih / h; A[i + 1, i] -= pih / h
    lam, Phi = eigh(A, np.diag(pi * h))
    lam[0] = 0.0; Phi[:, 0] = 1.0
    Phi = Phi / np.sqrt(np.sum(Phi**2 * (pi * h)[:, None], axis=0))
    dPhi = np.gradient(Phi, h, axis=0)
    cdf = np.cumsum(pi) * h; cdf /= cdf[-1]
    return dict(xg=xg, h=h, pi=pi, lam=lam, Phi=Phi, dPhi=dPhi, cdf=cdf)

def grid_basis_factory(G):
    xg, Phi, dPhi = G['xg'], G['Phi'], G['dPhi']; h = xg[1] - xg[0]
    def basis(x, r):
        idx = np.clip(np.searchsorted(xg, x), 1, len(xg) - 1)
        w = np.clip((x - xg[idx - 1]) / h, 0, 1)[:, None]
        P = (Phi[idx - 1, :r + 1] * (1 - w) + Phi[idx, :r + 1] * w).T
        dP = (dPhi[idx - 1, :r + 1] * (1 - w) + dPhi[idx, :r + 1] * w).T
        return P, dP
    def weighted(x, weights):
        # Interpolation is linear in the grid values; sum modes on the grid first.
        size = len(weights)
        return (np.interp(x, xg, Phi[:, :size] @ weights),
                np.interp(x, xg, dPhi[:, :size] @ weights))
    basis.weighted = weighted
    return basis

def bump_sampler(center, width, M, rng):
    u = np.linspace(center - width, center + width, 4001)
    dens = np.cos(np.pi * (u - center) / (2 * width))**2; F = np.cumsum(dens); F /= F[-1]
    return np.interp(rng.random(M), F, u)

# ---------------- metrics ----------------
def w2_gauss(X):
    M = len(X); q = norm.ppf((np.arange(M) + 0.5) / M)
    return np.sqrt(np.mean((np.sort(X) - q)**2))

def w2_grid(X, G):
    M = len(X); q = np.interp((np.arange(M) + 0.5) / M, G['cdf'], G['xg'])
    return np.sqrt(np.mean((np.sort(X) - q)**2))



In [ ]:
# code cell 3: common code of the reconstructed experiments: potentials, data, dictionaries (RBF / Legendre), reversible gEDMD estimator, exact 1D and 2D eigenpairs, transport, metrics
"""Common protocol for the reconstructed experiments (one code path).
Potentials: 'A' (double well in x1 + harmonic coordinates, nearest-neighbour coupling beta),
'quartic4', 'poly9' (full-plane separable polynomial multiwells). Dictionaries: 'rbf' and 'poly' (tensor Legendre).
Estimator: reversible gEDMD on the Dirichlet form (RR) with whitening; VAC lag eigenvalue for the disagreement diagnostic.
Exact eigenpairs: 1D grid (setup_1d) and 2D grid (sparse generalised eigenproblem of the Dirichlet form, Neumann)."""
import numpy as np, itertools, time
import scipy.sparse as sp, scipy.sparse.linalg as spla
from scipy.interpolate import RegularGridInterpolator
from numpy.polynomial import legendre as L
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import brentq
from functools import lru_cache

DTSIM, TAU = 2e-3, 0.1

# ------------------------------------------------------------------ potentials ------------------------------------------------------------------
class Potential:
    def __init__(self, kind, d=1, beta=0.0, k=2):
        self.kind, self.d, self.beta = kind, d, beta
        self.full_plane = kind in ('quartic4', 'poly9')
        self.reflect_box = False
        if self.full_plane:
            self.d = 2
            self.k = {'quartic4': 2, 'poly9': 3}[kind]
            self.centres = {2: np.array([-1., 1.]), 3: np.array([-1., 0., 1.])}[self.k]
            self.saddles = {2: np.array([0.]), 3: np.array([-1., 1.]) / np.sqrt(3)}[self.k]
            # This bulk sets grid/dictionary resolution, not the state space.
            bulk = brentq(lambda x: self.V1(x) - 30., self.centres[-1], 4.)
            self.box = np.array([[-bulk, bulk]] * 2)
        elif kind == 'A':
            self.k = k
            self.box = np.array([[-2.8, 2.8]] + [[-2.5, 2.5]] * (d - 1))
        else:
            raise ValueError(f'Unknown potential: {kind}')

    def V1(self, x):
        x = np.asarray(x)
        if self.kind == 'poly9': return (27. / 4.) * x**2 * (x**2 - 1)**2
        return (x**2 - 1)**2

    def grad1(self, x):
        if self.kind == 'poly9': return (27. / 2.) * x * (x**2 - 1) * (3 * x**2 - 1)
        return 4*x*(x**2 - 1)

    def hess1(self, x):
        if self.kind == 'poly9': return (27. / 2.) * (15*x**4 - 12*x**2 + 1)
        return 12*x**2 - 4

    def grad(self, X):
        if self.full_plane: return self.grad1(X)
        if self.kind == 'A':
            G = 2.0 * X.copy(); G[:, 0] = 4 * X[:, 0] * (X[:, 0]**2 - 1)
        else: G = 4 * X * (X**2 - 1)
        if self.beta > 0 and X.shape[1] > 1:
            D = X[:, 1:] - X[:, :-1]; G[:, :-1] -= self.beta * D; G[:, 1:] += self.beta * D
        return G

    def wells(self):
        c = self.centres if self.full_plane else np.array([-1., 1.])
        return np.array(list(itertools.product(c, c)))

    def reflect(self, X):
        # Full-plane B experiments never clip or reflect particles.
        if self.full_plane: return X
        return np.clip(X, self.box[:, 0], self.box[:, 1])

    def diffusion_step(self, X, rng, dt):
        if not self.full_plane:
            return self.reflect(X - self.grad(X)*dt + np.sqrt(2*dt)*rng.standard_normal(X.shape))
        # Drift-implicit Euler avoids explicit Euler explosions for superlinear
        # polynomial drift. It approximates the continuous diffusion (O(dt) bias).
        rhs = X + np.sqrt(2*dt)*rng.standard_normal(X.shape)
        Z = rhs.copy()
        for _ in range(30):
            delta = (Z + dt*self.grad1(Z) - rhs) / (1 + dt*self.hess1(Z))
            Z -= delta
            if np.max(np.abs(delta)) < 1e-12: break
        if not np.all(np.isfinite(Z)) or np.max(np.abs(Z + dt*self.grad1(Z) - rhs)) > 1e-9:
            raise RuntimeError('Implicit diffusion step did not converge; reduce dt_sim.')
        return Z


def _polynomial_target(kind, n=40001, tail_energy=40.):
    """Numerical full-line pi and saddle-basin masses, tensorised in 2D.
    Increase n and tail_energy to check quadrature and tail truncation errors.
    """
    pot = Potential(kind)
    bound = brentq(lambda x: pot.V1(x) - tail_energy, pot.centres[-1], 4.)
    x = np.linspace(-bound, bound, n); density = np.exp(-pot.V1(x))
    cdf = cumulative_trapezoid(density, x, initial=0); Z = cdf[-1]; cdf /= Z
    masses = np.diff(np.r_[0., np.interp(pot.saddles, x, cdf), 1.])
    return dict(x=x, cdf=cdf, Z=float(Z), mass1d=masses,
                mass2d=np.outer(masses, masses).ravel(),
                nearest_mass1d=np.diff(np.r_[0., np.interp((pot.centres[:-1]+pot.centres[1:])/2, x, cdf), 1.]),
                saddles=pot.saddles, barriers=pot.V1(pot.saddles),
                curvature=pot.hess1(pot.centres), bound=float(bound))


# Verified compatibility for the exact branch-removal edit below. Further
# edits to Potential receive a new version automatically.
_POTENTIAL_CACHE_COMPATIBILITY = {'25bb61c9d1ea502849831cfc6adb55862de649e7301af89550709dd85b3a8d04': 'a6fc5c3e8d7319e88fcf38c1465fb49d39224c7e37813fd30539591c0c5f6487'}

def potential_cache_version():
    actual = class_hash(Potential)
    return _POTENTIAL_CACHE_COMPATIBILITY.get(actual, actual)

def potential_key(pot):
    return dict(parameters=vars(pot), implementation=potential_cache_version())

def polynomial_target(kind, n=40001, tail_energy=40.):
    return cached_arrays('polynomial_target',
        dict(kind=kind, n=n, tail_energy=tail_energy,
             code=code_hash(_polynomial_target), potential=potential_cache_version()),
        lambda: _polynomial_target(kind, n, tail_energy))


def _simulate_pairs(pot, n, rng, chains=500, burn=20.0, init=None, dt_sim=None):
    d = pot.d; x = init(chains, rng) if init is not None else rng.uniform(pot.box[:, 0] * 0.5, pot.box[:, 1] * 0.5, size=(chains, d))
    dt_sim = DTSIM if dt_sim is None else dt_sim
    sub = max(1, int(round(TAU / dt_sim))); dt_sim = TAU / sub
    step = lambda x: pot.diffusion_step(x, rng, dt_sim)
    for _ in range(int(round(burn / dt_sim))): x = step(x)
    sub = int(round(TAU / dt_sim)); st = int(np.ceil(n / chains)); X = np.empty((st, chains, d)); Y = np.empty_like(X)
    for s in range(st):
        X[s] = x
        for _ in range(sub): x = step(x)
        Y[s] = x
    return X.reshape(-1, d)[:n], Y.reshape(-1, d)[:n]

def simulate_pairs(pot, n, rng, chains=500, burn=20.0, init=None, dt_sim=None):
    # Arbitrary initializer closures are intentionally not cached.
    if init is not None:
        return _simulate_pairs(pot, n, rng, chains, burn, init, dt_sim)
    key = dict(potential=potential_key(pot), n=n, chains=chains, burn=burn,
               dt_sim=DTSIM if dt_sim is None else dt_sim, tau=TAU,
               rng=rng.bit_generator.state, code=code_hash(_simulate_pairs))
    def compute():
        X, Y = _simulate_pairs(pot, n, rng, chains, burn, None, dt_sim)
        return dict(X=X, Y=Y, rng_after=rng.bit_generator.state)
    result = cached_arrays('trajectory_pairs', key, compute)
    # Consuming cached data advances the RNG exactly as a fresh simulation would.
    rng.bit_generator.state = result['rng_after']
    return result['X'], result['Y']


def stationary_samples(pot, M, rng, burn=40.0, thin=2.0):
    """B: independent grid-inverse-CDF target draws; A: long-run diffusion draws."""
    if pot.full_plane:
        target = polynomial_target(pot.kind)
        return np.interp(rng.random((M, pot.d)), target['cdf'], target['x'])
    X, _ = simulate_pairs(pot, M, rng, chains=M, burn=burn); return X

# ------------------------------------------------------------------ dictionaries ------------------------------------------------------------------
class Dictionary:
    """rbf: per-coordinate Gaussians (d=1), tensor grid (d=2), per-coordinate + neighbour-pair products (d>=3).
       poly: tensor Legendre polynomials of total degree <= p on the box rescaled to [-1,1]^d."""
    def __init__(self, kind, pot, n=12, ratio=None, p=12, wfrac=0.075):
        self.kind, self.pot, self.d = kind, pot, pot.d; box = pot.box
        if kind == 'rbf':
            self.cen = [np.linspace(box[i, 0], box[i, 1], n) for i in range(self.d)]
            self.wid = [(ratio * (self.cen[i][1] - self.cen[i][0])) if ratio is not None else wfrac * (box[i, 1] - box[i, 0]) for i in range(self.d)]
            if self.d == 2: self.pairs = None
            elif self.d >= 3: self.pairs = [(i, i + 1) for i in range(self.d - 1)]
        else:
            self.p = p; self.idx = [m for m in itertools.product(range(p + 1), repeat=self.d) if sum(m) <= p] if self.d <= 2 else \
                       [m for m in self._multi(self.d, p)]
    @staticmethod
    def _multi(d, p):
        out = []
        def rec(prefix, remaining, pos):
            if pos == d: out.append(tuple(prefix)); return
            for a in range(remaining + 1): rec(prefix + [a], remaining - a, pos + 1)
        rec([], p, 0); return out
    def size(self):
        if self.kind == 'poly': return len(self.idx)
        n = len(self.cen[0])
        return 1 + n if self.d == 1 else (1 + n * n if self.d == 2 else 1 + self.d * n + len(self.pairs) * n * n)
    def eval(self, X):
        """returns Psi (M,J) and dPsi (M,J,d)"""
        M, d = X.shape
        if self.kind == 'rbf':
            g = []; dg = []
            for i in range(d):
                z = (X[:, i][:, None] - self.cen[i][None, :]) / self.wid[i]; e = np.exp(-0.5 * z**2); g.append(e); dg.append(-z / self.wid[i] * e)
            if d == 1:
                Psi = np.hstack([np.ones((M, 1)), g[0]]); dPsi = np.hstack([np.zeros((M, 1)), dg[0]])[:, :, None]; return Psi, dPsi
            if d == 2:
                Psi = np.einsum('ma,mb->mab', g[0], g[1]).reshape(M, -1); dPx = np.einsum('ma,mb->mab', dg[0], g[1]).reshape(M, -1); dPy = np.einsum('ma,mb->mab', g[0], dg[1]).reshape(M, -1)
                return np.hstack([np.ones((M, 1)), Psi]), np.concatenate([np.zeros((M, 1, 2)), np.stack([dPx, dPy], 2)], 1)
            cols = [np.ones((M, 1))]; grads = [np.zeros((M, 1, d))]
            for i in range(d):
                cols.append(g[i]); G = np.zeros((M, g[i].shape[1], d)); G[:, :, i] = dg[i]; grads.append(G)
            for (i, j) in self.pairs:
                prod = np.einsum('ma,mb->mab', g[i], g[j]).reshape(M, -1); cols.append(prod)
                G = np.zeros((M, prod.shape[1], d)); G[:, :, i] = np.einsum('ma,mb->mab', dg[i], g[j]).reshape(M, -1); G[:, :, j] = np.einsum('ma,mb->mab', g[i], dg[j]).reshape(M, -1); grads.append(G)
            return np.hstack(cols), np.concatenate(grads, 1)
        # Legendre
        box = self.pot.box; Z = 2 * (X - box[:, 0]) / (box[:, 1] - box[:, 0]) - 1; scale = 2 / (box[:, 1] - box[:, 0])
        Pv = []; Pd = []
        for i in range(d):
            V = L.legvander(Z[:, i] if self.pot.full_plane else np.clip(Z[:, i], -1, 1), self.p); D = np.zeros_like(V)
            for k in range(1, self.p + 1): D[:, k] = L.legval(Z[:, i] if self.pot.full_plane else np.clip(Z[:, i], -1, 1), L.legder(np.eye(self.p + 1)[k]))
            Pv.append(V); Pd.append(D * scale[i])
        J = len(self.idx); Psi = np.ones((M, J)); dPsi = np.zeros((M, J, d))
        for j, m in enumerate(self.idx):
            for i in range(d): Psi[:, j] *= Pv[i][:, m[i]]
            for i in range(d):
                t = Pd[i][:, m[i]].copy()
                for l in range(d):
                    if l != i: t *= Pv[l][:, m[l]]
                dPsi[:, j, i] = t
        return Psi, dPsi

# ------------------------------------------------------------------ estimator ------------------------------------------------------------------
class RREstimate:
    def __init__(self, dic, Xd, Yd, tol=1e-8, chunk=5000, r_max=200, align=None):
        J = dic.size(); Gm = np.zeros((J, J)); Dm = np.zeros((J, J)); Am = np.zeros((J, J))
        for a in range(0, len(Xd), chunk):
            P, dP = dic.eval(Xd[a:a + chunk]); Q, _ = dic.eval(Yd[a:a + chunk]); Gm += P.T @ P; Am += P.T @ Q; Dm += np.einsum('mjd,mkd->jk', dP, dP, optimize=True)
        n = len(Xd); Gm /= n; Dm /= n; Am /= n
        sg, U = np.linalg.eigh(Gm); keep = sg > tol * sg.max(); W = U[:, keep] / np.sqrt(sg[keep])
        lw, Z = np.linalg.eigh(W.T @ Dm @ W); V = W @ Z; o = np.argsort(lw); lam, V = lw[o], V[:, o]; lam[0] = 0.0
        nrm = np.sqrt(np.einsum('jk,jk->k', V, Gm @ V)); V = V / nrm; V[:, 0] *= np.sign((Gm[0] @ V[:, 0]))
        self.dic, self.V, self.lam, self.kept, self.Gm, self.Am = dic, V, lam, int(keep.sum()), Gm, Am
        self.r_max = min(r_max, V.shape[1] - 1)
        ray = np.einsum('jk,jk->k', V, Am @ V) / np.einsum('jk,jk->k', V, Gm @ V); self.lam_lag = -np.log(np.clip(ray, 1e-12, None)) / TAU
        if align is not None:   # align signs with exact eigenfunctions evaluated on data
            Pex = align(Xd[:min(20000, n)]); P, _ = dic.eval(Xd[:min(20000, n)]); Ph = P @ V[:, :Pex.shape[0]]
            s = np.sign((Ph * Pex.T).sum(0)); s[s == 0] = 1; V[:, :Pex.shape[0]] *= s
    def basis(self, X, r):
        Vr = self.V[:, r] if isinstance(r, np.ndarray) else self.V[:, :r + 1]
        P, dP = self.dic.eval(X); return (P @ Vr).T, np.einsum('mjd,jk->kmd', dP, Vr)
    def disagreement(self, k): return abs(self.lam[k] - self.lam_lag[k]) / max(self.lam[k], 1e-12)

# Persist fitted eigenpairs/matrices separately from transport rank and horizon.
_rr_init = RREstimate.__init__
def _cached_rr_init(self, dic, Xd, Yd, tol=1e-8, chunk=5000, r_max=200, align=None):
    parameters = {k: v for k, v in vars(dic).items() if k != 'pot'}
    key = dict(dictionary=parameters, potential=potential_key(dic.pot),
        X=Xd, Y=Yd, tol=tol, chunk=chunk, tau=TAU,
        alignment=None if align is None else align(Xd[:min(20000, len(Xd))]),
        code=code_hash(_rr_init), dictionary_code=class_hash(Dictionary))
    def compute():
        fitted = object.__new__(RREstimate)
        _rr_init(fitted, dic, Xd, Yd, tol=tol, chunk=chunk, r_max=r_max, align=align)
        return {k: v for k, v in vars(fitted).items() if k not in ('dic', 'r_max')}
    self.__dict__.update(cached_arrays('rr_eigenpairs', key, compute))
    self.dic = dic
    self.r_max = min(r_max, self.V.shape[1] - 1)
RREstimate.__init__ = _cached_rr_init

# ------------------------------------------------------------------ exact eigenpairs ------------------------------------------------------------------
def exact_1d(kind='dw'):
    G = setup_1d(lambda x: (x**2 - 1)**2); return G

def _exact_2d_arrays(pot, n=201, kmax=40):
    """Grid eigenproblem. For full-plane B, Neumann endpoints approximate negligible tails at V1=30; they are not physical reflecting walls."""
    (ax, bx), (ay, by) = pot.box; xs = np.linspace(ax, bx, n); ys = np.linspace(ay, by, n); hx, hy = xs[1] - xs[0], ys[1] - ys[0]
    XX, YY = np.meshgrid(xs, ys, indexing='ij')
    if pot.full_plane: Vg = pot.V1(XX) + pot.V1(YY)
    else: Vg = (XX**2 - 1)**2 + YY**2 + 0.5 * pot.beta * (YY - XX)**2
    w = np.exp(-(Vg - Vg.min())); w /= w.sum() * hx * hy; N = n * n; idx = np.arange(N).reshape(n, n)
    rows, cols, vals = [], [], []
    def add(i, j, c):
        rows.extend([i, j, i, j]); cols.extend([i, j, j, i]); vals.extend([c, c, -c, -c])
    we = 0.5 * (w[1:, :] + w[:-1, :]) * hx * hy / hx**2; i, j = idx[1:, :].ravel(), idx[:-1, :].ravel(); add(i, j, we.ravel())
    we = 0.5 * (w[:, 1:] + w[:, :-1]) * hx * hy / hy**2; i, j = idx[:, 1:].ravel(), idx[:, :-1].ravel(); add(i, j, we.ravel())
    K = sp.csc_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(N, N)); Mm = sp.diags((w * hx * hy).ravel())
    lam, U = spla.eigsh(K, k=kmax + 1, M=Mm, sigma=-1e-3, which='LM'); o = np.argsort(lam); lam, U = lam[o], U[:, o]; lam[0] = 0.0
    U = U / np.sqrt(np.einsum('ij,i,ij->j', U, (w * hx * hy).ravel(), U)); U[:, 0] *= np.sign(U[:, 0].mean())
    Phi = U.reshape(n, n, -1); dPx = np.gradient(Phi, hx, axis=0); dPy = np.gradient(Phi, hy, axis=1)
    return dict(lam=lam, w=w, xs=xs, ys=ys, Phi=Phi, dPx=dPx, dPy=dPy, hx=hx, hy=hy)

def exact_2d(pot, n=201, kmax=40):
    data = cached_arrays('grid_2d',
        dict(potential=potential_key(pot), n=n, kmax=kmax, code=code_hash(_exact_2d_arrays)),
        lambda: _exact_2d_arrays(pot, n, kmax))
    xs, ys, Phi, dPx, dPy = (data[k] for k in ['xs', 'ys', 'Phi', 'dPx', 'dPy'])
    fP = RegularGridInterpolator((xs, ys), Phi, bounds_error=False, fill_value=None); fx = RegularGridInterpolator((xs, ys), dPx, bounds_error=False, fill_value=None); fy = RegularGridInterpolator((xs, ys), dPy, bounds_error=False, fill_value=None)
    def basis(X, r):
        # Bilinear interpolation commutes with selecting spectral columns.
        # Evaluate only requested modes; preserve linear extrapolation outside
        # the numerical bulk for full-plane polynomial targets.
        idx = r if isinstance(r, np.ndarray) else np.arange(r + 1)
        if len(idx) > 96:
            return fP(X)[:, idx].T, np.stack([fx(X)[:, idx], fy(X)[:, idx]], 2).transpose(1, 0, 2)
        ix = np.clip(np.searchsorted(xs, X[:, 0])-1, 0, len(xs)-2)
        iy = np.clip(np.searchsorted(ys, X[:, 1])-1, 0, len(ys)-2)
        tx = (X[:, 0]-xs[ix])/(xs[ix+1]-xs[ix])
        ty = (X[:, 1]-ys[iy])/(ys[iy+1]-ys[iy])
        P = np.zeros((len(X),len(idx)))
        G = np.zeros((len(idx),len(X),2))
        for a in (0,1):
            for b in (0,1):
                factor = (tx if a else 1-tx)*(ty if b else 1-ty)
                take = ((ix+a)[:,None],(iy+b)[:,None],idx)
                P += factor[:,None]*Phi[take]
                G[:,:,0] += (factor[:,None]*dPx[take]).T
                G[:,:,1] += (factor[:,None]*dPy[take]).T
        return np.ascontiguousarray(P.T), G

    def align_fn(X): return fP(X)[:, :5].T
    return dict(data, basis=basis, align=align_fn)

def energy_errors_2d(est, ex, Xs, kmax=4):
    """energy-norm errors ||grad(phi_hat_k - phi_k)||_{L2(pi)} by Monte Carlo over stationary samples Xs"""
    _, dPh = est.basis(Xs, kmax); _, dPe = ex['basis'](Xs, kmax)
    return np.sqrt(((dPh - dPe)**2).sum(-1).mean(1))[1:]

# ------------------------------------------------------------------ transport ------------------------------------------------------------------
def transport(X0, lam, basis, r, T, dt, pot, eps0=1e-3, vmax=20.0, r0=0, R=None, selection_source=None, diagnostics=None):
    """R = None: the first r modes in eigenvalue order. R > r: the r modes with the largest source coefficients among the first R (selection by coefficient); only the selected modes are evaluated along the transport."""
    from revision_common import PathMasks, selection_moments
    watch = PathMasks(len(X0)) if diagnostics is not None else None
    X = X0.copy()
    if R is None: idx = np.arange(r + 1)
    else:
        cand = np.asarray(R) if isinstance(R, np.ndarray) else np.arange(1, R + 1)     # candidate mode indices (R = int: 1..R)
        if selection_source is None:
            P0, _ = basis(X0, cand); cC = P0.mean(1)
        else:
            cC = selection_moments(basis, selection_source, cand)
        idx = np.concatenate([[0], cand[np.argsort(-np.abs(cC))[:r]]])
    P0, _ = basis(X0, idx); c0 = P0.mean(1); c0[0] = 1.0; lam_s = lam[idx]
    def vel(Y, t):
        w = np.exp(-lam_s * t) * c0
        fitted = getattr(basis, '__self__', None)
        if isinstance(fitted, RREstimate) and r0 == 0:
            # Linear spectral sums commute with dictionary evaluation. Avoid
            # materializing one gradient array for every retained eigenmode.
            coefficients = fitted.V[:, idx] @ w
            P, dP = fitted.dic.eval(Y)
            rho = P @ coefficients
            gradient = np.einsum('mjd,j->md', dP, coefficients)
        else:
            P, dP = basis(Y, idx)
            if r0 > 0: w[1:r0 + 1] = P[1:r0 + 1].mean(1)
            rho = w @ P
            gradient = np.einsum('k,kmd->md', w, dP)
        v = -gradient / np.maximum(rho, eps0)[:, None]
        nv = np.linalg.norm(v, axis=1, keepdims=True)
        if watch is not None: watch.velocity(rho,nv[:,0],eps0,vmax)
        return v * np.minimum(1.0, vmax / np.maximum(nv, 1e-12))
    def project(Y):
        result=pot.reflect(Y)
        if watch is not None:watch.projection(Y,result)
        return result
    for i in range(int(round(T / dt))):
        t = i * dt; k1 = vel(X, t); k2 = vel(project(X + .5 * dt * k1), t + .5 * dt); k3 = vel(project(X + .5 * dt * k2), t + .5 * dt); k4 = vel(project(X + dt * k3), t + dt)
        X = project(X + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4))
    if diagnostics is not None:
        diagnostics.update(watch.result(),selected_indices=idx.tolist(),initial_coefficients=c0.tolist(),
                           selection_sample_count=len(selection_source) if selection_source is not None else len(X0))
    return X

# ------------------------------------------------------------------ metrics ------------------------------------------------------------------
def sliced_w2(X, Y, seed=0, n=64):
    g = np.random.default_rng(seed); th = g.standard_normal((n, X.shape[1])); th /= np.linalg.norm(th, axis=1, keepdims=True)
    m = min(len(X), len(Y)); return float(np.sqrt(np.mean([np.mean((np.sort(X @ u)[:m] - np.sort(Y @ u)[:m])**2) for u in th])))
def energy_dist(X, Y, seed=0, m=1500):
    g = np.random.default_rng(seed); m = min(m, len(X), len(Y)); A, B = X[g.choice(len(X), m, replace=False)], Y[g.choice(len(Y), m, replace=False)]
    dd = lambda U, V: np.sqrt(((U[:, None, :] - V[None, :, :])**2).sum(-1)).mean(); return float(2 * dd(A, B) - dd(A, A) - dd(B, B))
def marg_w2(X, Y):
    m = min(len(X), len(Y)); return np.array([np.sqrt(np.mean((np.sort(X[:, i])[:m] - np.sort(Y[:, i])[:m])**2)) for i in range(X.shape[1])])
def well_masses(X, pot):
    if pot.full_plane:
        bins = np.searchsorted(pot.saddles, X[:, :2], side='right')
        return np.bincount(bins[:, 0]*pot.k + bins[:, 1], minlength=pot.k**2) / len(X)
    W = pot.wells(); idx = np.argmin(((X[:, None, :2] - W[None, :, :])**2).sum(-1), axis=1); return np.bincount(idx, minlength=len(W)) / len(X)
def w2_1d_exact(X, cdf, xg): return float(np.sqrt(np.mean((np.sort(X) - np.interp((np.arange(len(X)) + 0.5) / len(X), cdf, xg))**2)))


# Shared reference statistics and paired evaluation.
from scipy.spatial.distance import cdist

def energy_dist(X, Y, seed=0, m=1500):
    g = np.random.default_rng(seed); m = min(m, len(X), len(Y))
    A = X[g.choice(len(X), m, replace=False)]
    B = Y[g.choice(len(Y), m, replace=False)]
    return float(2*cdist(A, B).mean()-cdist(A, A).mean()-cdist(B, B).mean())

def particle_metrics(X, Y):
    if not np.isfinite(X).all(): raise FloatingPointError('Nonfinite transported particles')
    mm = marg_w2(X, Y)
    return dict(sw=float(sliced_w2(X, Y)), energy=energy_dist(X, Y),
                max_marginal=float(mm.max()), marginal_w2=mm.tolist())

def summarize(values):
    a = np.asarray(values, dtype=float)
    return dict(mean=np.mean(a, axis=0).tolist(),
                std=np.std(a, axis=0, ddof=1).tolist() if len(a)>1 else None,
                values=a.tolist())

REFERENCE_RESULTS = json.loads(Path('reference_statistics.json').read_text()) if Path('reference_statistics.json').exists() else {}

def reference_statistics(name, sampler_key, draw, M, gaussian=False, product=False):
    """Ten independent clouds (OU) or cloud pairs; fixed metric directions."""
    seeds = [[310000+2*b, 310001+2*b] for b in range(REFERENCE_REPEATS)]
    key = dict(sampler=sampler_key, M=M, gaussian=gaussian, product=product,
               repeats=REFERENCE_REPEATS, seeds=seeds, ddof=1,
               projection_seed=0, projections=64, energy_seed=0, energy_M=1500,
               code=code_hash(particle_metrics, energy_dist, sliced_w2, marg_w2, w2_gauss))
    def compute():
        rows = []
        for b, (s1,s2) in enumerate(seeds):
            A = draw(M, np.random.default_rng(s1))
            if gaussian:
                row = dict(w2=float(w2_gauss(A)))
            else:
                B = draw(M, np.random.default_rng(s2))
                row = particle_metrics(A, B)
                if product:
                    ca = np.bincount((A<0).sum(1), minlength=A.shape[1]+1)/M
                    cb = np.bincount((B<0).sum(1), minlength=B.shape[1]+1)/M
                    row['negative_count_tv'] = float(.5*np.abs(ca-cb).sum())
            rows.append(row)
            print(f'  reference {name}: {b+1}/{REFERENCE_REPEATS}', flush=True)
        return dict(repetitions=REFERENCE_REPEATS, seeds=seeds, ddof=1,
                    metrics={k:summarize([v[k] for v in rows]) for k in rows[0]},
                    configuration=_identity(key))
    value = cached_arrays('reference_statistics', key, compute)
    REFERENCE_RESULTS[name] = value
    Path('reference_statistics.json').write_text(json.dumps(REFERENCE_RESULTS, indent=2), encoding='utf-8')
    return value

def reference_for_potential(pot, M, name):
    return reference_statistics(name, dict(potential=potential_key(pot),
        sampler=code_hash(stationary_samples, _simulate_pairs, _polynomial_target),
        dt_sim=DTSIM, tau=TAU, burn=40.),
        lambda count, rng: stationary_samples(pot, count, rng), M)

def save_json(name, value):
    # Supplementary results are independent entries, outside the Part I loop.
    path = Path(name)
    if path.name == 'results.json' and isinstance(value, dict) and path.exists():
        existing = json.loads(path.read_text(encoding='utf-8'))
        supplementary = {key: item for key, item in existing.items()
                         if key not in value and (key in ('comparison', 'sample_size_table')
                                                  or key.startswith('A2v2_beta'))}
        value = dict(supplementary, **value)
    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')


In [ ]:
# code cell 4: Fixed double/multiwell experiments and appendix sensitivity studies
section(4, 'Double/multiwell sampling: fixed manuscript protocol',
        'RBF is the main learned method; Legendre is an appendix comparison. References: 10 independent repetitions, mean +/- sample SD.')
CONFIG = dict(version='manuscript-fixed-v1', quick=QUICK, frozen_n=FIXED_TRAINING_PAIRS,
              budgets=TRAINING_BUDGETS, M2D=M2D, M10D=M10D,
              seeds=[SEEDS_MAIN,SEEDS_SEC,SEEDS_10D], dt_a=DT_A2, dt_b=.02,
              tune=TUNE, r10=R10, references=REFERENCE_REPEATS,
              code=code_hash(transport, _rr_init, _simulate_pairs, _exact_2d_arrays),
              potential=potential_cache_version(), dictionary=class_hash(Dictionary))
R = dict(_config=CONFIG)
if Path('results.json').exists():
    previous = json.loads(Path('results.json').read_text())
    if previous.get('_config') == CONFIG:
        R = {name:case for name,case in previous.items() if name=='_config' or name in RETAINED_SYSTEMS}

for name,case in R.items():
    if not name.startswith('_'):
        case['presentation']='appendix limitation' if name in APPENDIX_SYSTEMS else 'main'

def source_A(pot, M, rng):
    return np.column_stack([bump_sampler(1.,.6,M,rng)] +
        [rng.standard_normal(M)/np.sqrt(2) for _ in range(pot.d-1)])

def make_dict(kind, pot, size=None):
    if kind=='rbf':
        n = size or (TUNE.get('rbf_n2d',12) if pot.d==2 else TUNE.get('rbf_n10d',8))
        return Dictionary(kind,pot,n=n,wfrac=TUNE['wfrac'])
    p = size or (TUNE['poly_p2d'] if pot.d==2 else TUNE.get('poly_p10d',3))
    return Dictionary('poly',pot,p=p)

def training_data(pot, seed):
    # Fixed maximal trajectory; all smaller budgets are exact prefixes.
    return simulate_pairs(pot, max(TRAINING_BUDGETS), np.random.default_rng(100+seed))

def fit_for(pot, kind, n, seed, align=None, size=None):
    Xd,Yd = training_data(pot,seed)
    dic = make_dict(kind,pot,size)
    chunk = 500 if pot.d==10 and kind=='poly' and dic.size()>500 else 2000 if pot.d==10 else 5000
    return RREstimate(dic, Xd[:n], Yd[:n], align=align, chunk=chunk)

def learned_transport(pot, est, X0, requested, dt, horizon=None, selection_source=None, diagnostics=None):
    cand = np.asarray([k for k in range(1,est.r_max+1) if est.disagreement(k)<.5],dtype=int) if pot.d==10 else None
    actual = min(requested, len(cand) if cand is not None else est.r_max)
    T = float(horizon if horizon is not None else 8./est.lam[1])
    if not np.isfinite(T) or T<=0: raise FloatingPointError('Invalid transport horizon')
    key = dict(potential=potential_key(pot), dictionary=vars(est.dic), X0=X0,
               V=est.V, lam=est.lam, candidates=cand, selection_source=selection_source, selection_rule=2, r=actual, T=T, dt=dt,
               code=code_hash(transport,Dictionary.eval,RREstimate.basis))
    key['dictionary'] = {k:v for k,v in vars(est.dic).items() if k!='pot'}
    def compute():
        info={}
        x=transport(X0,est.lam,est.basis,actual,T,dt,pot,R=cand,
                    selection_source=selection_source,diagnostics=info)
        return dict(final=x,diagnostics=info)
    result=cached_arrays('learned_transport',key,compute)
    X=result['final']
    if diagnostics is not None:diagnostics.update(result['diagnostics'])
    return X, actual, T

def evaluate_learned(pot, kind, n, seed, X0, Y, rank, dt, horizon=None, align=None, size=None):
    est = fit_for(pot,kind,n,seed,align,size)
    selection=source_A(pot,100000,np.random.default_rng(900+seed)) if pot.d==10 else None
    X,actual,T = learned_transport(pot,est,X0,rank,dt,horizon,selection_source=selection)
    row = dict(seed=seed, training_seed=100+seed, source_seed=500+seed,
               n_pairs=n,J=est.dic.size(),gram_rank=est.kept,r_requested=rank,
               r_used=actual,T=T,dt=dt,lambda1=float(est.lam[1]),
               finite=bool(np.isfinite(X).all()))
    if row['finite']:
        row.update(particle_metrics(X,Y))
        row['mass'] = well_masses(X,pot).tolist() if pot.d==2 else float((X[:,0]<0).mean())
    else:
        row.update(sw=None, failure='nonfinite output')
    return row



def summarize_method(rows):
    valid = all(v.get('sw') is not None and np.isfinite(v['sw']) for v in rows)
    return dict(rows=rows,sw=[v.get('sw') for v in rows],
                mean=float(np.mean([v['sw'] for v in rows])) if valid else None,
                std=float(np.std([v['sw'] for v in rows],ddof=1)) if valid and len(rows)>1 else None,
                all_finite=valid)

def run_system(pot,name,source,seeds):
    """Compatibility entry point to the fixed ten-realisation revision runners."""
    if seeds!=10:raise ValueError('The retained revision protocol requires ten realisations')
    from revision_notebook import ensure_stages
    from revision_publish import publish_regular,publish_a2
    global R
    stage='a2' if pot.kind=='A' and pot.d==2 else 'a10' if pot.d==10 else 'multi'
    ensure_stages(stage)
    R=json.loads(Path('results.json').read_text())
    if stage=='a2':publish_a2(R)
    else:publish_regular(R,stage)
    save_json('results.json',R)

# All repeated cells use the fixed revision runners, including independent 10D selection.
from revision_notebook import ensure_stages
from revision_publish import publish_regular
ensure_stages('multi','a10')
R=json.loads(Path('results.json').read_text())
publish_regular(R,'multi');publish_regular(R,'a10')
save_json('results.json',R)


## Part II: OU and independent products
Analytical OU spectrum; initial-particle coefficients. All reference statistics use 10 independent repetitions.


In [ ]:
# code cell 6: Part II settings
section(6,'OU and independent-product settings','OU uses analytical spectra and initial-particle coefficients; references use 10 repetitions.')
SC=dict(M=200,Mbig=300,n_pairs=3000,dt=.05,seeds=1) if QUICK else dict(M=10000,Mbig=20000,n_pairs=200000,dt=.05,seeds=1)
results={}


## OU library
Hermite eigenpairs and fixed empirical initial coefficients.


In [ ]:
# code cell 7: OU analytical Hermite spectrum and fixed particle coefficients
section(7,'OU analytical spectrum and initial-particle coefficients')

def ou_coefficients(X0,r):
    P,_=hermite_basis(X0,r); c=P.mean(axis=1); c[0]=1.
    return c

def ou_velocity(X,t,c,watch=None):
    r=len(c)-1; P,dP=hermite_basis(X,r)
    weights=np.exp(-np.arange(r+1)*t)*c
    rho=weights@P; drho=weights@dP
    raw=-drho/np.maximum(rho,1e-3)
    if watch is not None:watch.velocity(rho,np.abs(raw))
    return np.clip(raw,-20.,20.)

def ou_transport(X0,r,T,dt=.05,diagnostics=None):
    """Same initial cloud estimates coefficients and is then transported."""
    from revision_common import PathMasks
    watch=PathMasks(len(X0)) if diagnostics is not None else None
    c=ou_coefficients(X0,r); X=X0.copy()
    for i in range(int(round(T/dt))):
        t=i*dt; k1=ou_velocity(X,t,c,watch)
        k2=ou_velocity(X+.5*dt*k1,t+.5*dt,c,watch)
        k3=ou_velocity(X+.5*dt*k2,t+.5*dt,c,watch)
        k4=ou_velocity(X+dt*k3,t+dt,c,watch)
        X+=dt/6*(k1+2*k2+2*k3+k4)
    if diagnostics is not None:diagnostics.update(watch.result())
    return X

def ou_run(X0,r,T):
    return cached_arrays('ou_particle_transport',dict(X0=X0,r=r,T=T,dt=.05,
        code=code_hash(ou_transport,ou_coefficients,ou_velocity,hermite_basis)),
        lambda:ou_transport(X0,r,T,.05))

def ou_reference(M):
    return reference_statistics(f'OU_M{M}',dict(target='N(0,1)',sampler='numpy.standard_normal'),
        lambda count,rng:rng.standard_normal(count),M,gaussian=True)

def ou_population_parameters(t):
    # Analytical density is used only as a reference, never to set coefficients.
    return 1.5*np.exp(-t), .25*np.exp(-2*t)+1-np.exp(-2*t)

OU=dict(config=dict(quick=QUICK,M=SC['M'],dt=.05,source_mean=1.5,source_std=.5,
    source_seed=0,coefficient_method='initial-particle empirical means',
    spectrum='analytical Hermite',reference_repetitions=REFERENCE_REPEATS),
    rank_scan=[],particle_scan=[],time_scan=[])

def save_ou(): save_json('ou_results.json',OU)


## OU spectral-rank scan
Initial particles follow N(1.5,0.5^2); the target is N(0,1). Retained plotted ranks: 10,15,20,30,60. Path diagnostics additionally include r=40.


In [ ]:
# code cell 8: OU retained-mode scan
section(8,'OU sampling: increasing analytical spectral rank',
        'Initial coefficients are empirical means over the transported source particles.')
X0_ou=1.5+.5*np.random.default_rng(0).standard_normal(SC['M'])
OU['reference']=ou_reference(SC['M'])
for rank in ([10,20,30,60] if QUICK else [10,15,20,30,60]):
    X=ou_run(X0_ou,rank,6.)
    row=dict(r=rank,w2=float(w2_gauss(X)),M=len(X),T=6.,source_seed=0)
    OU['rank_scan'].append(row);save_ou();print(row,flush=True)


## OU transported-particle-count scan
Use analytical eigenpairs, r=40, T=6, and coefficients estimated from each initial cloud.
Full-mode particle counts: 500, 1000, 2000, 5000, 20000.


In [ ]:
# code cell 9: OU transported-particle-count scan
section(9,'OU sampling: increasing transported-particle count',
        'Analytical Hermite spectrum, r=40, T=6, initial-particle coefficients.')
for M in ([500,2000,SC['M']] if QUICK else [500,1000,2000,5000,20000]):
    X0=1.5+.5*np.random.default_rng(1).standard_normal(M)
    X=ou_run(X0,40,6.)
    row=dict(M=M,r=40,T=6.,source_seed=1,w2=float(w2_gauss(X)),reference=ou_reference(M))
    OU['particle_scan'].append(row);save_ou()
    print(f'OU M={M}: W2={row["w2"]:.6f}; reference={row["reference"]["metrics"]["w2"]}',flush=True)


## OU terminal-time scan
Use analytical eigenpairs and the same empirical initial coefficients for T=1,2,3,4,6.


In [ ]:
# code cell 11: OU terminal-time scan with initial-particle coefficients
section(11,'OU sampling: increasing terminal time',
        'Analytical spectrum; r=40; coefficients estimated once from initial particles.')
for T in [1.,2.,3.,4.,6.]:
    X=ou_run(X0_ou,40,T)
    mean,variance=ou_population_parameters(T)
    row=dict(T=T,r=40,M=len(X),source_seed=0,w2=float(w2_gauss(X)),
             population_gaussian_w2=float(np.sqrt(mean**2+(np.sqrt(variance)-1)**2)))
    OU['time_scan'].append(row);save_ou();print(row,flush=True)


## Basic reversible 10D Ornstein-Uhlenbeck sampling

We use $dX_t=-AX_t\,dt+\sqrt{2}\,dW_t$, with $A=I+0.15E^TE$ and $(Ex)_j=x_{j+1}-x_j$.
The invariant law is explicitly $\pi=\mathcal N(0,A^{-1})$; neighboring coordinates are coupled.
If $A=Q\operatorname{diag}(a_j)Q^T$, set $z_j=\sqrt{a_j}\,q_j^Tx$.
For the nonnegative operator $L=-\Delta+(Ax)\cdot\nabla$, analytical eigenpairs are $\phi_{\boldsymbol n}=\prod_j\operatorname{He}_{n_j}(z_j)/\sqrt{n_j!}$ and $\lambda_{\boldsymbol n}=\sum_j n_j a_j$.

This first high-dimensional OU check uses the known factorization in the eigenbasis, with 30 nonconstant Hermite modes per coordinate and fixed coefficients estimated from the initial particles. It does not construct a general joint multivariate truncation. Full mode uses 5,000 particles, ten source seeds, step 0.05, and ten independent target-reference repetitions. The source is $\mathcal N(0.5\mathbf1,0.25I)$.

The matched zero-coupling baseline sets A=I and has target N(0,I). Both cases use identical source clouds, time steps, ranks, projection directions and source/reference seed lists.


In [ ]:
# code cell 12: Coupled and uncoupled reversible 10D OU with analytical Hermite eigenpairs
section(12,'Reversible 10D OU sampling with analytical eigenpairs')
from scipy.special import ndtri

def ou_hd_model(d=10, coupling=.15):
    edges=np.zeros((d-1,d))
    for j in range(d-1):edges[j,j]=-1.;edges[j,j+1]=1.
    A=np.eye(d)+coupling*(edges.T@edges)
    rates,Q=np.linalg.eigh(A)
    # Fix the otherwise arbitrary signs of the eigenvectors for reproducibility.
    Q*=np.where(Q[np.argmax(np.abs(Q),axis=0),np.arange(d)]<0,-1.,1.)
    covariance=(Q/rates)@Q.T
    return A,rates,Q,covariance

def ou_hd_coefficients(Z0,rank):
    c=np.ones((Z0.shape[1],rank+1))
    previous=np.ones_like(Z0);current=Z0.copy()
    for k in range(1,rank+1):
        if k>1:previous,current=current,(Z0*current-np.sqrt(k-1)*previous)/np.sqrt(k)
        c[:,k]=current.mean(axis=0)
    return c

def ou_hd_spectral_field(Z,t,c,rates):
    weights=c*np.exp(-rates[:,None]*np.arange(c.shape[1])[None,:]*t)
    rho=np.broadcast_to(weights[:,0],Z.shape).copy();gradient=np.zeros_like(Z)
    previous=np.ones_like(Z);current=Z.copy()
    for k in range(1,c.shape[1]):
        if k>1:previous,current=current,(Z*current-np.sqrt(k-1)*previous)/np.sqrt(k)
        rho+=current*weights[:,k]
        gradient+=np.sqrt(k)*previous*weights[:,k]
    # In whitened eigen-coordinates, the diffusion mobility is diag(rates).
    velocity=-rates*gradient/np.maximum(rho,1e-3)
    return velocity,rho

def ou_hd_transport(X0,rates,Q,rank,times,dt=.05):
    from revision_common import PathMasks
    watch=PathMasks(X0.shape)
    started=time.perf_counter()
    Z=(X0@Q)*np.sqrt(rates)
    coefficients=ou_hd_coefficients(Z,rank)
    checkpoints={int(round(t/dt)):i for i,t in enumerate(times)}
    states=np.empty((len(times),)+X0.shape);states[checkpoints[0]]=X0
    diagnostics=dict(coordinate_stage_evaluations=0,negative_density=0,density_floor=0,
                     velocity_cap=0,minimum_density_ratio=1.)
    def velocity(Y,t):
        v,rho=ou_hd_spectral_field(Y,t,coefficients,rates)
        if not np.isfinite(v).all():raise FloatingPointError('Nonfinite 10D OU velocity')
        watch.velocity(rho,np.abs(v))
        diagnostics['coordinate_stage_evaluations']+=rho.size
        diagnostics['negative_density']+=int(np.count_nonzero(rho<0))
        diagnostics['density_floor']+=int(np.count_nonzero(rho<1e-3))
        diagnostics['velocity_cap']+=int(np.count_nonzero(np.abs(v)>20))
        diagnostics['minimum_density_ratio']=min(diagnostics['minimum_density_ratio'],float(rho.min()))
        return np.clip(v,-20.,20.)
    for step in range(max(checkpoints)):
        t=step*dt;k1=velocity(Z,t)
        k2=velocity(Z+.5*dt*k1,t+.5*dt)
        k3=velocity(Z+.5*dt*k2,t+.5*dt)
        k4=velocity(Z+dt*k3,t+dt)
        Z+=dt*(k1+2*k2+2*k3+k4)/6
        if step+1 in checkpoints:
            states[checkpoints[step+1]]=(Z/np.sqrt(rates))@Q.T
    diagnostics.update(watch.result())
    return dict(states=states,coefficients=coefficients,diagnostics=diagnostics,
                transport_seconds=time.perf_counter()-started)

def ou_hd_gaussian_w2_columns(values,target_std,target_mean=0.):
    """Exact empirical-to-Gaussian 1D W2 per column, using quantile integrals."""
    sorted_values=np.sort(np.asarray(values)-target_mean,axis=0)
    count=len(sorted_values)
    edges=ndtri(np.arange(count+1)/count)
    pdf=np.exp(-.5*edges**2)/np.sqrt(2*np.pi)
    quantile_integrals=pdf[:-1]-pdf[1:]
    std=np.asarray(target_std)
    squared=np.mean(sorted_values**2,axis=0)+std**2-2*std*(quantile_integrals@sorted_values)
    return np.sqrt(np.maximum(squared,0.))

def ou_hd_metrics(X,covariance,directions):
    projected_std=np.sqrt(np.einsum('ki,ij,kj->k',directions,covariance,directions))
    projected=ou_hd_gaussian_w2_columns(X@directions.T,projected_std)
    marginal=ou_hd_gaussian_w2_columns(X,np.sqrt(np.diag(covariance)))
    empirical_covariance=np.cov(X,rowvar=False,ddof=1)
    adjacent=np.diag(np.corrcoef(X,rowvar=False),1)
    return dict(sw2=float(np.sqrt(np.mean(projected**2))),marginal_w2=marginal.tolist(),
                mean_marginal_w2=float(marginal.mean()),max_marginal_w2=float(marginal.max()),
                mean_norm=float(np.linalg.norm(X.mean(axis=0))),
                covariance_relative_error=float(np.linalg.norm(empirical_covariance-covariance)/np.linalg.norm(covariance)),
                adjacent_correlations=adjacent.tolist())

def ou_hd_summary(values):
    values=np.asarray(values,dtype=float)
    return dict(mean=values.mean(axis=0).tolist(),
                std=values.std(axis=0,ddof=1).tolist() if len(values)>1 else None,
                values=values.tolist())

def ou_hd_references(covariance,Q,rates,directions,count,seeds):
    rows=[]
    for seed in seeds:
        X=(np.random.default_rng(seed).standard_normal((count,len(rates)))/np.sqrt(rates))@Q.T
        rows.append(ou_hd_metrics(X,covariance,directions))
    return dict(repetitions=len(seeds),seeds=seeds,ddof=1,
                comparison='Each independent target cloud against the analytical Gaussian target',
                metrics={key:ou_hd_summary([row[key] for row in rows]) for key in rows[0]})

def run_ou_hd_case(OU_HD_CONFIG,results_path,plot_path,reference_name):
    print(f"10D OU coupling = {OU_HD_CONFIG['coupling']:g}",flush=True)
    ou_hd_started=time.perf_counter()
    HD_A,HD_RATES,HD_Q,HD_COV=ou_hd_model(OU_HD_CONFIG['dimension'],OU_HD_CONFIG['coupling'])
    HD_DIRECTIONS=np.random.default_rng(OU_HD_CONFIG['projection_seed']).standard_normal((64,10))
    HD_DIRECTIONS/=np.linalg.norm(HD_DIRECTIONS,axis=1,keepdims=True)
    hd_checks=dict(lyapunov_residual=float(np.linalg.norm(HD_A@HD_COV+HD_COV@HD_A.T-2*np.eye(10))),
                   orthogonality_residual=float(np.linalg.norm(HD_Q.T@HD_Q-np.eye(10))))
    assert hd_checks['lyapunov_residual']<1e-12
    # Independent checks of the Gaussian metric and the transformed velocity scaling.
    assert np.allclose(ou_hd_gaussian_w2_columns(np.array([[2.,-3.]]),np.array([1.,2.])),np.sqrt([5.,13.]))
    hd_probe=np.random.default_rng(77).normal(size=(20,10))
    hd_c=np.zeros((10,3));hd_c[:,0]=1.;hd_c[:,1]=.03;hd_c[:,2]=.02
    hd_v,hd_rho=ou_hd_spectral_field(hd_probe,.4,hd_c,HD_RATES)
    hd_x=(hd_probe/np.sqrt(HD_RATES))@HD_Q.T
    hd_grad=np.empty_like(hd_x);hd_eps=1e-5
    for j in range(10):
        plus=hd_x.copy();minus=hd_x.copy();plus[:,j]+=hd_eps;minus[:,j]-=hd_eps
        rp=ou_hd_spectral_field((plus@HD_Q)*np.sqrt(HD_RATES),.4,hd_c,HD_RATES)[1]
        rm=ou_hd_spectral_field((minus@HD_Q)*np.sqrt(HD_RATES),.4,hd_c,HD_RATES)[1]
        hd_grad[:,j]=(np.log(rp).sum(axis=1)-np.log(rm).sum(axis=1))/(2*hd_eps)
    hd_checks['velocity_gradient_residual']=float(np.max(np.abs((hd_v/np.sqrt(HD_RATES))@HD_Q.T+hd_grad)))
    assert hd_checks['velocity_gradient_residual']<1e-8

    hd_reference=cached_arrays('ou_hd_reference',dict(particles=OU_HD_CONFIG['particles'],seeds=OU_HD_CONFIG['reference_seeds'],covariance=HD_COV,
        rates=HD_RATES,Q=HD_Q,directions=HD_DIRECTIONS,
        code=code_hash(ou_hd_references,ou_hd_metrics,ou_hd_gaussian_w2_columns,ou_hd_summary)),
        lambda:ou_hd_references(HD_COV,HD_Q,HD_RATES,HD_DIRECTIONS,
            OU_HD_CONFIG['particles'],OU_HD_CONFIG['reference_seeds']))
    hd_rows=[];hd_first=None
    for seed in OU_HD_CONFIG['source_seeds']:
        print(f'10D OU: transporting source seed {seed}',flush=True)
        X0=OU_HD_CONFIG['source_mean']+OU_HD_CONFIG['source_std']*np.random.default_rng(seed).standard_normal((OU_HD_CONFIG['particles'],10))
        trajectory=cached_arrays('ou_hd_transport',dict(X0=X0,rates=HD_RATES,Q=HD_Q,
            rank=OU_HD_CONFIG['nonconstant_modes_per_coordinate'],times=OU_HD_CONFIG['times'],dt=.05,
            code=code_hash(ou_hd_transport,ou_hd_coefficients,ou_hd_spectral_field)),
            lambda:ou_hd_transport(X0,HD_RATES,HD_Q,OU_HD_CONFIG['nonconstant_modes_per_coordinate'],OU_HD_CONFIG['times'],.05))
        metrics=[dict(time=t,**ou_hd_metrics(X,HD_COV,HD_DIRECTIONS))
                 for t,X in zip(OU_HD_CONFIG['times'],trajectory['states'])]
        hd_rows.append(dict(seed=seed,metrics=metrics,diagnostics=trajectory['diagnostics'],
                            transport_seconds=trajectory['transport_seconds']))
        if hd_first is None:hd_first=trajectory['states']
        print(f"10D OU seed {seed}: SW2 {metrics[0]['sw2']:.6f} -> {metrics[-1]['sw2']:.6f}; transport {trajectory['transport_seconds']:.1f}s",flush=True)
    hd_time_summary=[]
    for i,t in enumerate(OU_HD_CONFIG['times']):
        hd_time_summary.append(dict(time=t,metrics={key:ou_hd_summary([row['metrics'][i][key] for row in hd_rows])
            for key in hd_rows[0]['metrics'][i] if key!='time'}))
    hd_target_corr=HD_COV/np.sqrt(np.outer(np.diag(HD_COV),np.diag(HD_COV)))
    OU_HD=dict(config=OU_HD_CONFIG,drift_matrix=HD_A.tolist(),rates=HD_RATES.tolist(),
        eigenvectors=HD_Q.tolist(),target_covariance=HD_COV.tolist(),
        target_adjacent_correlations=np.diag(hd_target_corr,1).tolist(),
        reference=hd_reference,rows=hd_rows,time_summary=hd_time_summary,implementation_checks=hd_checks,
        elapsed_seconds=time.perf_counter()-ou_hd_started,
        interpretation='Basic correlated Gaussian benchmark. The known reversible OU rotation makes the source and target factorize. Rank is per coordinate, not a global multivariate mode count.')
    save_json(results_path,OU_HD)
    REFERENCE_RESULTS[reference_name]=hd_reference
    save_json('reference_statistics.json',REFERENCE_RESULTS)
    np.savez_compressed(plot_path,source=hd_first[0],transported=hd_first[-1],
                        target_covariance=HD_COV,times=np.asarray(OU_HD_CONFIG['times']))
    print('10D OU target reference:',hd_reference['metrics']['sw2'],flush=True)
    print('10D OU complete in',round(OU_HD['elapsed_seconds'],1),'seconds',flush=True)
    return OU_HD

# Analytical spectra, ten paired source clouds, unchanged target references.
from revision_notebook import ensure_stages
from revision_publish import publish_ou
ensure_stages('ou');publish_ou()
OU_HD=json.loads(Path('ou_hd_results.json').read_text())
OU_HD_ZERO=json.loads(Path('ou_hd_uncoupled_results.json').read_text())


## Independent double-well products
10D and 50D targets have 2^d wells. Coordinate-wise FD-1401 and Koopman (RBF) are main comparisons; Koopman (Legendre) (degree 32, J=33) is retained in the appendix. Full mode uses M=20000 and 200000 training pairs. Each coordinate retains up to 16 nonconstant modes.

The labels Koopman (RBF) and Koopman (Legendre) denote generator eigenpairs estimated by reversible gEDMD with the respective dictionaries. They specify the spectral approximation used by the same BKT transport.

Ten prescribed repetitions use indices s=0--5,7--10, source seed 1+s and training seed 1001+s; the target seed stays 7. Index 6 is excluded because source and target would share seed 7; index 10 uses source 11 and training 1011. Histograms display the first repetition, while marginal-error panels show ten-repetition means and sample standard deviations.


In [ ]:
# code cell 15: Independent product of double wells in $d$ dimensions
section(15, 'Independent product of double wells in $d$ dimensions', 'Independent product of one-dimensional double wells $V(x)=\\sum_{i=1}^d(x_i^2-1)^2$, whose invariant law has $2^d$ wells, in $d=10$ and $d=50$. (a) Product case: the transport acts on each coordinate separately, with FD-1401 one-dimensional eigenpairs and with Koopman (RBF) and Koopman (Legendre) eigenpairs estimated per coordinate from the same $d$-dimensional trajectories; the source is a product of bumps in the right well of every coordinate, concentrated in one corner. QUICK uses M=300 and 3000 pairs in both dimensions; full mode uses M=20000 and 200000 training pairs.')
from copy import deepcopy
# Embedded coordinate-wise Legendre gEDMD implementation.
"""One-dimensional reversible gEDMD with a Legendre dictionary.

Used independently on each coordinate of the separable product experiments.
No simulations or experiments are executed when this module is imported.
"""
import numpy as np
from numpy.polynomial import legendre as L


def fit_legendre_1d(samples, grid, grid_eigenfunctions, degree=32,
                    bounds=(-2.8, 2.8), tol=1e-10, r_max=24, chunk=5000):
    """Fit D a = lambda G a from stationary samples and dictionary gradients.

    P_0,...,P_degree includes the constant: J=degree+1 per coordinate.
    The affine map sends bounds to [-1,1]. Polynomials and their derivatives
    are evaluated without clipping the argument, including outside bounds.
    FD eigenfunctions are used only for sign alignment, as in the RBF branch.
    """
    samples = np.asarray(samples, dtype=float).reshape(-1)
    lower, upper = bounds
    if degree < 1 or upper <= lower or len(samples) == 0 or chunk < 1:
        raise ValueError('Invalid Legendre degree, bounds, samples, or chunk size.')
    scale = 2.0 / (upper - lower)
    J = degree + 1
    gram = np.zeros((J, J))
    stiffness = np.zeros_like(gram)
    for start in range(0, len(samples), chunk):
        z = scale * (samples[start:start + chunk] - lower) - 1.0
        P = L.legvander(z, degree)
        dP = np.zeros_like(P)
        dP[:, 1] = 1.0
        for k in range(2, J):
            dP[:, k] = (2 * k - 1) * P[:, k - 1] + dP[:, k - 2]
        dP *= scale
        gram += P.T @ P
        stiffness += dP.T @ dP
    gram /= len(samples)
    stiffness /= len(samples)
    sg, U = np.linalg.eigh(gram)
    keep = sg > tol * sg.max()
    if np.count_nonzero(keep) < 2:
        raise ValueError('Legendre whitening retained no nonconstant modes.')
    W = U[:, keep] / np.sqrt(sg[keep])
    reduced = W.T @ stiffness @ W
    lam, Z = np.linalg.eigh((reduced + reduced.T) / 2)
    V = W @ Z
    lam[0] = 0.0
    V /= np.sqrt(np.einsum('jk,jk->k', V, gram @ V))
    sign = np.sign(gram[0] @ V[:, 0])
    V[:, 0] *= sign if sign else 1.0
    available = min(r_max, V.shape[1] - 1)
    align_count = min(available + 1, grid_eigenfunctions.shape[1])
    cross = np.zeros(align_count)
    for start in range(0, len(samples), chunk):
        x = samples[start:start + chunk]
        learned = L.legval(scale * (x - lower) - 1, V[:, :align_count])
        reference = np.stack([np.interp(x, grid, grid_eigenfunctions[:, k])
                              for k in range(align_count)])
        cross += np.sum(learned * reference, axis=1)
    signs = np.sign(cross)
    signs[signs == 0] = 1
    V[:, :align_count] *= signs
    coefficients = V[:, :available + 1].copy()
    derivatives = L.legder(coefficients, axis=0) * scale

    def basis(x, r):
        if r < 0 or r > available:
            raise ValueError(f'Requested {r} modes, but only {available} are available.')
        z = scale * (np.asarray(x) - lower) - 1
        return (L.legval(z, coefficients[:, :r + 1]),
                L.legval(z, derivatives[:, :r + 1]))

    def weighted(x, weights):
        z = scale * (np.asarray(x) - lower) - 1
        size = len(weights)
        return (L.legval(z, coefficients[:, :size] @ weights),
                L.legval(z, derivatives[:, :size] @ weights))
    basis.weighted = weighted

    metadata = dict(dictionary='Legendre', degree=int(degree), J=J,
                    bounds=list(bounds), whitening_tolerance=tol,
                    gram_rank=int(keep.sum()), available_modes=int(available),
                    chunk=int(chunk))
    return lam[:available + 1], basis, metadata


# Per-coordinate dictionary; no high-dimensional tensor dictionary is built.
PRODUCT_WORKERS = 8  # independent coordinates; BLAS remains single-threaded
PRODUCT_LEGENDRE_DEGREE = 32  # J=33, including the constant
PRODUCT_LEGENDRE_TOL = 1e-10
PRODUCT_LEGENDRE_CHUNK = 5000

Gp = setup_1d(lambda x: (x**2 - 1)**2); basis_p = grid_basis_factory(Gp); xgp, lam_p, Phi_p, cdf_p = Gp['xg'], Gp['lam'], Gp['Phi'], Gp['cdf']
def sample_pi_product(M, d, g): return np.column_stack([np.interp(g.random(M), cdf_p, xgp) for _ in range(d)])
def neg_count_dist(X, d): return np.bincount((X < 0).sum(1), minlength=d + 1) / len(X)
def rr_eig_1d(Xd, cen, wd, tol=1e-10, r_max=24):
    def f(x):
        delta = np.asarray(x)[:, None] - cen
        values = np.exp(-delta**2 / (2 * wd**2))
        return (np.hstack([np.ones((len(x), 1)), values]),
                np.hstack([np.zeros((len(x), 1)), -delta / wd**2 * values]))
    P, dP = f(Xd); Gm = P.T @ P / len(Xd); Dm = dP.T @ dP / len(Xd)
    sg, U = np.linalg.eigh(Gm); keep = sg > tol * sg.max(); W = U[:, keep] / np.sqrt(sg[keep]); lw, Z = np.linalg.eigh(W.T @ Dm @ W); V = W @ Z
    o = np.argsort(lw); lh, V = lw[o], V[:, o]; lh[0] = 0.0; nrm = np.sqrt(((P @ V)**2).mean(axis=0)); V = V / nrm; V[:, 0] *= np.sign((P @ V[:, 0]).mean())
    rm = min(r_max, V.shape[1] - 1); Pex = np.stack([np.interp(Xd, xgp, Phi_p[:, k]) for k in range(rm + 1)]); s_ = np.sign(np.sum((P @ V[:, :rm + 1]) * Pex.T, axis=0)); s_[s_ == 0] = 1; V[:, :rm + 1] *= s_
    def basis(x, rr):
        P, dP = f(x)
        return (P @ V[:, :rr + 1]).T, (dP @ V[:, :rr + 1]).T
    def weighted(x, weights):
        x = np.asarray(x)
        coefficients = V[:, :len(weights)] @ weights
        values = np.exp(-(x[:, None]-cen)**2/(2*wd**2))
        density = values @ coefficients[1:]
        gradient = (values @ (coefficients[1:]*cen)-x*density)/wd**2
        return coefficients[0]+density, gradient
    basis.weighted = weighted
    return lh, basis
def transport_product(X0, lams, bases, r, T, dt, floor=1e-3, vmax=20., workers=None,diagnostics=None):
    # The potential and velocity are coordinate-wise separable. Each worker
    # integrates one full coordinate path with the same RK4 steps and clipping.
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from revision_common import PathMasks
    M, d = X0.shape
    watch=PathMasks((M,d),scope='Coordinate RK stages; clipping only at completed steps') if diagnostics is not None else None
    ranks = np.full(d, r, dtype=int) if np.isscalar(r) else np.asarray(r, dtype=int)
    if ranks.shape != (d,): raise ValueError("Expected one retained rank per coordinate.")
    def coordinate(i):
        rank = int(ranks[i]); basis = bases[i]
        P0, _ = basis(X0[:, i], rank)
        coefficients = P0.mean(axis=1); coefficients[0] = 1.0
        decay = lams[i][:rank + 1]
        x = X0[:, i].copy()
        local=PathMasks(M) if watch is not None else None
        def velocity(y, t):
            weights = np.exp(-decay * t) * coefficients
            if hasattr(basis, 'weighted'):
                rho, drho = basis.weighted(y, weights)
            else:
                P, dP = basis(y, rank); rho, drho = weights @ P, weights @ dP
            raw=-drho / np.maximum(rho, floor)
            if local is not None:local.velocity(rho,np.abs(raw),floor,vmax)
            return np.clip(raw, -vmax, vmax)
        for k in range(int(round(T / dt))):
            t = k * dt
            k1 = velocity(x, t)
            k2 = velocity(x + .5 * dt * k1, t + .5 * dt)
            k3 = velocity(x + .5 * dt * k2, t + .5 * dt)
            k4 = velocity(x + dt * k3, t + dt)
            proposed=x + dt / 6 * (k1 + 2*k2 + 2*k3 + k4)
            x = np.clip(proposed, xgp[0], xgp[-1])
            if local is not None:local.projection(proposed,x)
        if watch is not None:
            for name in local.masks:watch.masks[name][:,i]=local.masks[name]
        return x
    workers = max(1, min(d, PRODUCT_WORKERS if workers is None else workers))
    X = np.empty_like(X0)
    if workers == 1:
        for i in range(d): X[:, i] = coordinate(i)
    else:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(coordinate, i): i for i in range(d)}
            for completed, future in enumerate(as_completed(futures), 1):
                X[:, futures[future]] = future.result()
                if completed % 10 == 0 or completed == d:
                    print(f"      coordinate transports complete: {completed}/{d}", flush=True)
    if diagnostics is not None:diagnostics.update(watch.result())
    return X

def sde_pairs_d(n, d, tau, g, drift, chains=500, burn=20.0, dt=2e-3):
    x = sample_pi_product(chains, d, g)
    for _ in range(int(burn / dt)): x = x + drift(x) * dt + np.sqrt(2 * dt) * g.standard_normal(x.shape)
    sub = int(round(tau / dt)); st = int(np.ceil(n / chains)); X = np.empty((st, chains, d)); Y = np.empty_like(X)
    for s_ in range(st):
        X[s_] = x
        for _ in range(sub): x = x + drift(x) * dt + np.sqrt(2 * dt) * g.standard_normal(x.shape)
        Y[s_] = x
    return X.reshape(-1, d)[:n], Y.reshape(-1, d)[:n]
# Ten sources, source seed 1+s and training seed 1001+s; shared fixed target.
from revision_notebook import ensure_stages
from revision_publish import publish_products
ensure_stages('product10','product50');publish_products()


## Appendix: OU density and stability diagnostics
Truncated density ratios and particle activity use empirical initial coefficients. Gaussian population formulas serve only as references.


In [ ]:
# code cell 17: Appendix OU density and stability diagnostics
section(17,'Appendix: OU truncated density and stability',
        'Analytical spectrum and empirical initial coefficients; population Gaussian density is only a diagnostic reference.')
b=4.; xg=np.linspace(-b,b,3201); h=xg[1]-xg[0]
diagnostics=dict(config=dict(M=SC['M'],source_seed=0,dt=.05,T=6.,bulk=[-b,b],
    coefficient_method='initial-particle empirical means',particle_integrator='RK4'),rows=[])
for r in [10,20,40]:
    c=ou_coefficients(X0_ou,r); P,dP=hermite_basis(xg,r); lam=np.arange(r+1)
    minima=[]
    for t in [0.,.1,.5,1.]: minima.append(float((np.exp(-lam*t)*c@P).min()))
    L_all=L_good=0.; minmass=1.; burn=None
    X=X0_ou.copy(); good_steps=floor_good=clip_good=0
    floor_all=clip_all=negative_all=0
    for i in range(120):
        t=i*.05; w=np.exp(-lam*t)*c; rh=w@P; drh=w@dP
        mean,variance=ou_population_parameters(t)
        rex=norm.pdf(xg,mean,np.sqrt(variance))/norm.pdf(xg)
        eps=float(np.max(np.abs(rh-rex)))
        if burn is None and np.all(rh>=.5*rex): burn=t
        v=np.clip(-drh/np.maximum(rh,1e-3),-20.,20.); dv=np.gradient(v,h)
        good_grid=rex>=np.sqrt(eps)
        L_all+=float(dv.max())*.05
        L_good+=(float(dv[good_grid].max()) if good_grid.any() else 0.)*.05
        if good_grid.any(): minmass=min(minmass,float(norm.cdf(xg[good_grid].max(),mean,np.sqrt(variance))-norm.cdf(xg[good_grid].min(),mean,np.sqrt(variance))))
        if r in (20,40):
            PX,dPX=hermite_basis(X,r); rho=w@PX; grad=w@dPX
            exact_ratio=norm.pdf(X,mean,np.sqrt(variance))/norm.pdf(X)
            good=(np.abs(X)<=b)&(exact_ratio>=2*eps)
            floor=rho<1e-3; clip=np.abs(grad/np.maximum(rho,1e-3))>20
            good_steps+=int(good.sum()); floor_good+=int((good&floor).sum()); clip_good+=int((good&clip).sum())
            floor_all+=int(floor.sum()); clip_all+=int(clip.sum()); negative_all+=int((rho<0).sum())
            k1=ou_velocity(X,t,c); k2=ou_velocity(X+.025*k1,t+.025,c)
            k3=ou_velocity(X+.025*k2,t+.025,c); k4=ou_velocity(X+.05*k3,t+.05,c)
            X+=.05/6*(k1+2*k2+2*k3+k4)
    row=dict(r=r,density_ratio_minima=minima,times=[0.,.1,.5,1.],burn_in=burn,
        lipschitz_integral_bulk=L_all,lipschitz_integral_good=L_good,min_good_mass=minmass)
    if r in (20,40): row.update(good_particle_step_fraction=good_steps/(120*len(X)),
        floor_good=floor_good,clip_good=clip_good,floor_all=floor_all,clip_all=clip_all,
        negative_all=negative_all,final_w2=float(w2_gauss(X)))
    diagnostics['rows'].append(row);print(row,flush=True)
save_json('ou_diagnostics.json',diagnostics)


## Figures and presentation settings

All plotting cells are grouped below, after the numerical experiments. To redraw from the saved results in a fresh kernel, run **Figure data and style** and then the desired figure cell. These cells read `outputs/` and do not run simulations or estimate spectra.

In **Figure data and style**, `PAPER_FONT_SIZES` in `supplementary/figure_style.py` defines printed font sizes: overall titles 11 pt, panel titles 9.5 pt, axis labels 9 pt, ticks 8.5 pt and legends 8 pt. Each manuscript figure converts these sizes using its canvas width and the current LaTeX insertion width. Regenerate the affected PDFs after changing those widths. Export the full canvas without tight cropping so this conversion stays exact. `FIGURE_SIZES` and the individual plotting cells control geometry; these presentation controls are separate from the numerical settings.

| Figure cell | Output files |
|---|---|
| Admissible-source rank and coefficient figures | `fig_admissible_source_ranks.pdf`, `fig_admissible_source_coefficients.pdf` |
| Matched OU paths | `fig_ou_path_comparison.pdf` |
| Equal-data comparison | `fig_data_comparison.pdf` |
| Alanine distribution and convergence | `fig_alanine.pdf`, `fig_alanine_convergence.pdf` |
| One-dimensional OU figure | `fig_ou.pdf` |
| Ten-dimensional OU time and correlation figure | `fig_ou10.pdf` |
| Ten-dimensional OU marginal-comparison figure | `fig_ou10_marginals.pdf` |
| Independent-product figures in 10D and 50D | `fig_hd10_dictionaries.pdf`, `fig_hd50_dictionaries.pdf` |

Table export and report publication occur together in the final publication cell, after all prescribed tasks have completed. Run numerical cells only when new simulation results are needed.

### Alanine dipeptide figure notes

`outputs/figures/fig_alanine.pdf` uses the same 256-mode BKT runs as `fig_alanine_convergence.pdf`, at normalized flow time $s=\lambda_1 T=8$. The three angle panels display 1,000 source, transported and reference particles for seed 1101. The distance panel includes all three paired source/reference designs (seeds 1101-1103); each line connects a transported-to-reference distance with a reference-to-reference distance. The dashed line and band give the reference mean and sample standard deviation. This band is descriptive, not a confidence interval or an independent-sample error floor.

The spectrum is estimated by a reversible Dirichlet-form calculation with Fourier degree 28 (3,249 basis functions), retaining 256 nonconstant modes, smoothing 0.10 rad and Gram cutoff 1e-12. These are eigenpairs of an auxiliary reversible diffusion determined by the smoothed angle distribution, not estimates of physical molecular-dynamics kinetics. Source particles are scrambled Sobol points transformed to a fixed local wrapped Gaussian. BKT fixes the empirical initial spectral coefficients, while LAWGD updates the same moments along the particle paths. Both methods use the same spectrum, initial clouds and normalized time scale. KDE uses the same target-estimation frames and 0.10 rad smoothing. Variability across the three designs does not quantify uncertainty across independent MD datasets.

The distribution figure has no overall title or subtitle. Both figures use the shared printed typography and legend spacing. Run the **Alanine distribution and convergence figures** cell below to redraw both PDFs from saved results. The distribution figure can also be regenerated with `python -B supplementary/alanine_experiment.py --stage plot`; use `--stage verify` to check its saved metrics or `--stage reproduce --seed 1101 --refit` to refit the spectrum and reproduce the illustrative 256-mode BKT run. Use `python -B supplementary/alanine_convergence.py --stage verify` to check all saved convergence checkpoints. All results and limitations are in [the single shared report](outputs/all_experiment_results_and_assessment.md#alanine-dipeptide). The RBF collapse diagnostic is retained and reproduced with `python -B supplementary/alanine_experiment.py --case rbf --stage reproduce`.


In [ ]:
# Figure data and style: run this cell before any figure cell, including after a kernel restart.
from pathlib import Path
import os,json
import numpy as np
import matplotlib.pyplot as plt

_plot_here=Path.cwd().resolve()
PLOT_PROJECT_DIR=_plot_here.parent if _plot_here.name=='outputs' else _plot_here
import sys
sys.path.insert(0, str(PLOT_PROJECT_DIR / "supplementary"))
from experiment_store import activate_notebook_store
activate_notebook_store()
PLOT_OUTPUT_DIR=PLOT_PROJECT_DIR/'outputs'
os.chdir(PLOT_OUTPUT_DIR)
PLOT_PART1=json.loads(Path('results.json').read_text())
PLOT_OU=json.loads(Path('ou_results.json').read_text())
PLOT_PRODUCTS={}
with np.load('product_plot_data.npz',allow_pickle=False) as _plot_data:
    for _plot_dimension in [10,50]:
        for _plot_method in ['exact','estimated','legendre']:
            _plot_key=f'pdw_{_plot_dimension}_{_plot_method}'
            PLOT_PRODUCTS[_plot_key]=tuple(_plot_data[f'{_plot_key}_{field}'].copy() for field in
                ['dimension','marginal_w2','reference_marginal_w2','source_x1','transported_x1','target_x1'])

# Shared printed typography: compensate for each manuscript insertion width.
from figure_style import FONT_SIZES as PLOT_FONT_SIZES, configure_style
FIGURE_SIZES={
    'ou_1d':(19,11),
    'ou_10d':(14.5,6.5),
    'ou_10d_marginals':(14,11.5),
    'products_appendix':(17,11.2),
}

from matplotlib.ticker import NullFormatter, MaxNLocator, LogLocator, FuncFormatter

# Shared presentation settings for every saved figure; no numerical settings change.
configure_style()

PLOT_COLORS = {'fd': '#0072B2', 'rbf': '#D55E00', 'poly': '#009E73',
               'reference': '#525252', 'particles': '#0072B2'}


def finish_axis(ax, log_y=False):
    if log_y:
        ax.set_yscale('log')
        low, high = ax.get_ylim()
        ratio = high/low
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4,steps=[1,2,5,10]) if ratio<1.5 else
            LogLocator(base=10,subs=(1,2,3,5) if ratio<10 else (1,2,5) if ratio<50 else (1,)))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda value,pos: f'{value:g}'))
        ax.yaxis.set_minor_formatter(NullFormatter())
    ax.grid(axis='y', which='major', color='#E0E0E0', linewidth=.8)
    ax.set_axisbelow(True)
    ax.tick_params(which='major', length=6, width=1)

LEGEND_GAP_PT = 32
LEGEND_BOTTOM_MARGIN_PT = 18

def plot_content_bottom(fig, axes, renderer):
    """Lowest rendered plot content, including labels, ticks and common labels."""
    bounds=[ax.get_tightbbox(renderer) for ax in np.asarray(axes).ravel() if ax.get_visible()]
    bounds += [text.get_window_extent(renderer) for text in fig.texts
               if text.get_visible() and text.get_text()]
    return min(box.y0 for box in bounds if box is not None)

def shared_legend(fig, axes, ncol=None):
    entries = {}
    for ax in np.asarray(axes).ravel():
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels):
            entries.setdefault(label, handle)
    order = ['FD-201','FD-1401 factor','Koopman (RBF)','Koopman (Legendre)']
    labels = [label for label in order if label in entries]+[label for label in entries if label not in order]
    legend=fig.legend([entries[label] for label in labels],labels,loc='upper center',
               bbox_to_anchor=(.5,0),bbox_transform=fig.transFigure,
               ncol=ncol or len(entries),frameon=False,borderpad=0,borderaxespad=0,
               handlelength=2.1,columnspacing=1.35,handletextpad=.6)
    # Place every legend a fixed physical distance below the nearest content.
    # Reserve extra footer space when a two-row legend needs more room.
    for _ in range(3):
        fig.canvas.draw()
        renderer=fig.canvas.get_renderer()
        px_per_pt=fig.dpi/72
        content_bottom=plot_content_bottom(fig,axes,renderer)
        legend_height=legend.get_window_extent(renderer).height
        needed_bottom=(LEGEND_GAP_PT+LEGEND_BOTTOM_MARGIN_PT)*px_per_pt+legend_height
        shift=max(0,needed_bottom-content_bottom)/fig.bbox.height
        if shift<1e-6:break
        fig.subplots_adjust(bottom=fig.subplotpars.bottom+shift)
        common_xlabel=fig._supxlabel
        if common_xlabel is not None:
            x,y=common_xlabel.get_position()
            common_xlabel.set_position((x,y+shift))
    fig.canvas.draw()
    renderer=fig.canvas.get_renderer()
    top=(plot_content_bottom(fig,axes,renderer)-LEGEND_GAP_PT*fig.dpi/72)/fig.bbox.height
    legend.set_bbox_to_anchor((.5,top),transform=fig.transFigure)
    return legend

def reference_band(ax,x,refs):
    means=np.array([v['mean'] for v in refs]);sd=np.array([v['std'] for v in refs])
    ax.errorbar(x,means,yerr=sd,fmt='D:',color=PLOT_COLORS['reference'],
                markersize=6,capsize=4,label='Target reference',zorder=2)

def reference_interval(ax, ref):
    # The report defines these intervals as mean +/- one standard deviation.
    ax.axhspan(max(ref['mean']-ref['std'],np.finfo(float).tiny),
               ref['mean']+ref['std'],color=PLOT_COLORS['reference'],alpha=.10,
               linewidth=0,zorder=0)
    ax.axhline(ref['mean'],color=PLOT_COLORS['reference'],ls=':',
               label='Target reference',zorder=1)


In [ ]:
# One-dimensional OU figure
# Extra width keeps the closely spaced rank ticks horizontal and legible.
configure_style('fig_ou', FIGURE_SIZES['ou_1d'][0])
fig,axes=plt.subplots(1,3,figsize=FIGURE_SIZES['ou_1d'],gridspec_kw={'width_ratios':[1.15,1,1]})
ax=axes[0];rank_x=[v['r'] for v in PLOT_OU['rank_scan']]
ax.plot(rank_x,[v['w2'] for v in PLOT_OU['rank_scan']],'o-',
        color=PLOT_COLORS['particles'],label='Transported particles')
ref=PLOT_OU['reference']['metrics']['w2'];reference_interval(ax,ref)
ax.set_xticks([10,30,60]);ax.set_xticks([15,20],minor=True)
ax.tick_params(axis='x',labelrotation=0)
ax.set_xlabel('Modes, $r$');ax.set_title('Spectral\nrank')
ax=axes[1];rows=PLOT_OU['particle_scan'];x=[v['M'] for v in rows]
ax.plot(x,[v['w2'] for v in rows],'o-',color=PLOT_COLORS['particles'],label='Transported particles')
reference_band(ax,x,[v['reference']['metrics']['w2'] for v in rows])
ax.set_xscale('log');ax.set_xticks([500,20000],['500','20k']);ax.set_xticks([2000],minor=True)
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel('Particles, $M$');ax.set_title('Particle\ncount')
ax=axes[2];time_x=[v['T'] for v in PLOT_OU['time_scan']]
ax.plot(time_x,[v['w2'] for v in PLOT_OU['time_scan']],'o-',
        color=PLOT_COLORS['particles'],label='Transported particles')
reference_interval(ax,ref)
ax.set_xticks([1,3,6]);ax.set_xticks([2,4],minor=True);ax.set_xlabel('Time, $T$');ax.set_title('Transport\ntime')
for col,ax in enumerate(axes):
    ax.set_ylabel('$W_2$' if col==0 else '')
    finish_axis(ax,log_y=True);ax.margins(y=.12,x=.12)
axes[0].yaxis.set_major_locator(LogLocator(base=10,subs=(1,2,5)))
fig.suptitle('Ornstein-Uhlenbeck sampling',y=.97)
fig.subplots_adjust(left=.16,right=.985,top=.68,bottom=.28,wspace=.70)
shared_legend(fig,axes)
fig.savefig('figures/fig_ou.pdf');plt.close(fig)


In [ ]:
# Ten-dimensional OU time and correlation figure
hd=json.loads(Path('ou_hd_results.json').read_text())
configure_style('fig_ou10', FIGURE_SIZES['ou_10d'][0])
fig,axes=plt.subplots(1,2,figsize=FIGURE_SIZES['ou_10d'])
particle_color=PLOT_COLORS['particles'];target_color='#009E73'
ax=axes[0];times=[row['time'] for row in hd['time_summary']]
values=[row['metrics']['sw2'] for row in hd['time_summary']]
ax.errorbar(times,[v['mean'] for v in values],yerr=[v['std'] or 0 for v in values],
            fmt='o-',capsize=4,color=particle_color,label='Transported particles')
reference_interval(ax,hd['reference']['metrics']['sw2'])
ax.set_xlabel('Transport time, $T$');ax.set_ylabel('Sliced $W_2$')
ax.set_xticks([0,1,2,3,4,6]);ax.set_title('Transport time');finish_axis(ax,log_y=True)
ax=axes[1];coordinates=np.arange(1,hd['config']['dimension'])
correlation=hd['time_summary'][-1]['metrics']['adjacent_correlations']
ax.errorbar(coordinates,correlation['mean'],yerr=correlation['std'],fmt='o-',
            capsize=4,color=particle_color,label='Transported particles')
refcorr=hd['reference']['metrics']['adjacent_correlations']
ax.fill_between(coordinates,np.asarray(refcorr['mean'])-refcorr['std'],
                np.asarray(refcorr['mean'])+refcorr['std'],color=PLOT_COLORS['reference'],alpha=.1)
ax.plot(coordinates,hd['target_adjacent_correlations'],'--',color=target_color,label='Analytical target')
ax.set_xticks([1,3,5,7,9]);ax.set_xlabel('Coordinate pair $(j,j+1)$');ax.set_ylabel('Correlation')
ax.set_title('Coordinate correlations');finish_axis(ax)
fig.suptitle('Ornstein-Uhlenbeck sampling in 10 dimensions',y=.97)
fig.subplots_adjust(left=.115,right=.975,top=.77,bottom=.28,wspace=.40)
shared_legend(fig,axes)
fig.savefig('figures/fig_ou10.pdf');plt.close(fig)


In [ ]:
# Ten-dimensional OU marginal-comparison figure
from scipy.stats import gaussian_kde

configure_style('fig_ou10_marginals', FIGURE_SIZES['ou_10d_marginals'][0])
fig,axes=plt.subplots(2,2,figsize=FIGURE_SIZES['ou_10d_marginals'],sharex='col',sharey='row')
grid=np.linspace(-3.5,3.5,141)
gx,gy=np.meshgrid(grid,grid)
points=np.stack([gx,gy],axis=-1)
density_levels=np.linspace(0,.21,22)
target_color='#009E73'
for col,(suffix,coupling) in enumerate([('_uncoupled',0.),('',.15)]):
    with np.load(f'ou_hd{suffix}_plot_data.npz') as data:
        source=data['source'];transported=data['transported'];covariance=data['target_covariance']
    ax=axes[0,col]
    bins=np.linspace(-3.5,3.5,61)
    ax.hist(source[:,0],bins=bins,density=True,color='#B6B6B6',alpha=.6,label='Initial particles')
    ax.hist(transported[:,0],bins=bins,density=True,color=PLOT_COLORS['particles'],alpha=.6,label='Transported particles')
    variance=covariance[0,0]
    ax.plot(grid,np.exp(-grid**2/(2*variance))/np.sqrt(2*np.pi*variance),
            color=target_color,label='Analytical target')
    ax.set_title('Marginal $x_1$')
    ax.set_xlabel('$x_1$');ax.set_ylabel('Density' if col==0 else '')
    ax.tick_params(axis='x',labelbottom=True)
    finish_axis(ax)
    ax.grid(False)
    ax=axes[1,col]
    # A genuine marginal uses the covariance submatrix; other coordinates are integrated out.
    marginal_covariance=covariance[:2,:2]
    inverse=np.linalg.inv(marginal_covariance)
    peak=1/(2*np.pi*np.sqrt(np.linalg.det(marginal_covariance)))
    exact_joint=peak*np.exp(-.5*np.einsum('...i,ij,...j->...',points,inverse,points))
    # KDE is only a visualization of the final samples; sampling and metrics use no KDE.
    kde=gaussian_kde(transported[:,:2].T,bw_method='scott')
    sampled_joint=kde(points.reshape(-1,2).T).reshape(gx.shape)
    filled=ax.contourf(gx,gy,sampled_joint,levels=density_levels,cmap='Blues',vmin=0,vmax=.21,extend='max')
    ax.contour(gx,gy,sampled_joint,levels=peak*np.array([.01,.10,.50]),
               colors=[PLOT_COLORS['particles']],linewidths=1.5)
    ax.contour(gx,gy,exact_joint,levels=peak*np.array([.01,.10,.50]),
               colors=[target_color],linestyles='--',linewidths=2.2)
    ax.set_title('Joint marginal $(x_1,x_2)$')
    ax.set_xlabel('$x_1$');ax.set_ylabel('$x_2$' if col==0 else '')
    ax.set_aspect('equal',adjustable='box')
    ax.set_xlim(-3.5,3.5);ax.set_ylim(-3.5,3.5)
    ax.set_xticks([-3,0,3]);ax.set_yticks([-3,0,3])
    ax.tick_params(which='major',length=6,width=1)
# One subtitle describes the whole figure; the in-panel key applies to all panels.
fig.suptitle('Marginal distributions of the 10D OU process',y=.985)
fig.text(.5,.925,r'Left: $\gamma=0$;  Right: $\gamma=0.15$',
         ha='center',va='center',fontsize=PLOT_FONT_SIZES['panel'])
column_lefts=[.09,.565]
column_width=.385
joint_height=column_width*fig.get_figwidth()/fig.get_figheight()
for col,left in enumerate(column_lefts):
    axes[0,col].set_position([left,.705,column_width,.15])
    axes[1,col].set_position([left,.09,column_width,joint_height])
handles,_=axes[0,0].get_legend_handles_labels()
axes[0,0].legend(handles,['Initial','Transported','Target'],
                 loc='upper left',frameon=False,fontsize=PLOT_FONT_SIZES['legend'],handlelength=1.4,
                 handletextpad=.5,borderaxespad=.25,labelspacing=.35)
# Identical joint-density levels are retained across columns, without a colorbar.
fig.savefig('figures/fig_ou10_marginals.pdf')
plt.close(fig)


In [ ]:
# Independent-product figures in 10D and 50D
for d in [10,50]:
    kinds=['exact','estimated','legendre']
    configure_style(f'fig_hd{d}_dictionaries', FIGURE_SIZES['products_appendix'][0])
    fig,axes=plt.subplots(2,len(kinds),figsize=FIGURE_SIZES['products_appendix'],
                          squeeze=False,sharey='row')
    product=json.loads(Path(f'product_{d}_results.json').read_text())
    for col,kind in enumerate(kinds):
        _,mw,mwr,source,X,Y=PLOT_PRODUCTS[f'pdw_{d}_{kind}']
        label={'exact':'FD-1401', 'estimated':'Koopman (RBF)',
               'legendre':'Koopman (Legendre)'}[kind]
        ax=axes[0,col]
        ax.hist(source,bins=np.linspace(-3.5,3.5,71),density=True,
                color='#B6B6B6',alpha=.6,label='Initial particles')
        ax.hist(X,bins=np.linspace(-3.5,3.5,71),density=True,
                color=PLOT_COLORS['particles'],alpha=.6,label='Transported particles')
        ax.hist(Y,bins=np.linspace(-3.5,3.5,71),density=True,histtype='step',
                color='#252525',linewidth=2,label='Target samples')
        ax.set_title(label);ax.set_xlabel('Coordinate $x_1$')
        ax.set_ylabel('Density' if col==0 else '')
        ax.set_xlim(-3.5,3.5);ax.set_xticks([-3,0,3])
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        finish_axis(ax)
        ax=axes[1,col];x=np.arange(1,d+1)
        ax.errorbar(x,mw,yerr=product['methods'][kind]['metrics']['marginal_w2']['std'],
                fmt='o-',markersize=6 if d==10 else 4,capsize=3 if d==10 else 2,
                color=PLOT_COLORS['particles'],label='Transported particles')
        sd=product['reference']['metrics']['marginal_w2']['std']
        ax.errorbar(x,mwr,yerr=sd,fmt=':',color=PLOT_COLORS['reference'],
                    capsize=3 if d==10 else 2,label='Target reference')
        ax.set_xlabel('Coordinate');ax.set_ylabel('Marginal $W_2$' if col==0 else '')
        ax.set_xticks([1,5,10] if d==10 else [1,25,50])
        if d==50:ax.set_xticks([10,20,30,40],minor=True)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        finish_axis(ax);ax.margins(x=.04,y=.12)
    fig.suptitle('Independent double-well sampling',y=.975)
    fig.text(.5,.88,f'{d}-dimensional product',ha='center',fontsize=PLOT_FONT_SIZES['panel'])
    fig.subplots_adjust(left=.14,right=.98,
                        top=.77,bottom=.19,wspace=.15,hspace=.55)
    shared_legend(fig,axes,ncol=2)
    fig.savefig(f'figures/fig_hd{d}_dictionaries.pdf');plt.close(fig)


In [ ]:
# Alanine distribution and convergence figures: saved results only.
from pathlib import Path
import sys
from IPython.display import display, FileLink
_alanine_here = Path.cwd().resolve()
_alanine_root = _alanine_here.parent if _alanine_here.name == 'outputs' else _alanine_here
sys.path.insert(0, str(_alanine_root / 'supplementary'))
from alanine_experiment import plot as plot_alanine_distribution
from alanine_convergence import plot as plot_alanine_convergence
plot_alanine_distribution()
plot_alanine_convergence()
for _name in ('fig_alanine', 'fig_alanine_convergence'):
    display(FileLink((_alanine_root / 'outputs/figures' / (_name + '.pdf')).relative_to(_alanine_here).as_posix()))


## Tables and unified report
The final publication cell validates all prescribed tasks and publishes the tables and the single results report together. It requires all prescribed results to be present.


In [ ]:
# Link to the last completely published report; the final publication cell refreshes it.
from IPython.display import display, FileLink
display(FileLink('all_experiment_results_and_assessment.md'))


In [ ]:
# Ten paired equal-data repetitions; metrics and timings come from fresh serial runs.
from revision_notebook import ensure_stages
from revision_publish import publish_a2,publish_comparison
ensure_stages('a2','comparison');publish_a2();publish_comparison()
COMPARISON_RESULTS=json.loads(Path('data_comparison/summary.json').read_text())
for row in COMPARISON_RESULTS['rows']:
    print(row['beta'],row['label'],row['sw2_mean'],row['sw2_std'],row['hit_count'],row['n_repeats'])


In [ ]:
# Equal-available-data comparison figure: saved results only.
import subprocess
from pathlib import Path
from IPython.display import display, FileLink
_comparison_plot_here = Path.cwd().resolve()
_comparison_plot_project = _comparison_plot_here.parent if _comparison_plot_here.name == 'outputs' else _comparison_plot_here
subprocess.run(['uv', 'run', '--with', 'numpy==2.4.6', '--with', 'matplotlib==3.11.1',
                'python', 'supplementary/plot_data_comparison.py'], cwd=_comparison_plot_project, check=True)
_comparison_pdf = _comparison_plot_project / 'outputs/figures/fig_data_comparison.pdf'
display(FileLink(_comparison_pdf.relative_to(_comparison_plot_here).as_posix()))


In [ ]:
# Admissible-source double wells: fixed ten-seed protocol and per-path diagnostics.
from revision_notebook import ensure_stages
from revision_publish import publish_a2
ensure_stages('a2');publish_a2()
ADMISSIBLE_SOURCE_RESULTS=json.loads(Path('admissible_source/summary.json').read_text())


In [ ]:
# Admissible-source figures and tables: saved results only.
import subprocess
from pathlib import Path
from IPython.display import display, FileLink
_admissible_plot_here = Path.cwd().resolve()
_admissible_plot_root = _admissible_plot_here.parent if _admissible_plot_here.name == 'outputs' else _admissible_plot_here
subprocess.run(['uv', 'run', '--with', 'numpy==2.4.6', '--with', 'matplotlib==3.11.1',
                'python', 'supplementary/plot_admissible_source.py'], cwd=_admissible_plot_root, check=True)
for _name in ('fig_admissible_source_ranks', 'fig_admissible_source_coefficients'):
    _admissible_pdf = _admissible_plot_root / 'outputs/figures' / (_name + '.pdf')
    display(FileLink(_admissible_pdf.relative_to(_admissible_plot_here).as_posix()))



## Fixed-configuration costs and evaluation
Run the fixed-fit alanine cost, held-out evaluation and source-design checks, retain every prescribed outcome, and publish the unified report after all retained experiments are complete.


In [ ]:
from threadpoolctl import threadpool_limits
from experiment_store import activate_notebook_store, close_notebook_store
from revision_notebook import ensure_stages
from revision_publish import main as publish_revision
from revision_verify import main as verify_revision
from revision_figures import main as draw_revision_figures
activate_notebook_store()
try:
    ensure_stages('alanine_cost','alanine_variability','alanine_iid')
    with threadpool_limits(limits=1):
        verify_revision()
        publish_revision()
        draw_revision_figures()
finally:
    close_notebook_store()
